# `C14` — Capacity edges, one part per ceiling

Scenario **C14** of the Track A certification ladder (card `OMIE-CERT-C14-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C14; plan [`scenario-ladder-plan.md`](../../docs/omiecid/scenario-ladder-plan.md) §C14 and §The capacity-validity precondition). Oracle kind: **exact-parametric on documented behaviour, plus the battery with per-part exclusions, each asserted expected-violated**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. A *ceiling* is one of the simulator's five fixed-array or loop bounds; a *part* is one ceiling's construction; the *full side* is the side of the book a part fills. Every engine-side assertion is an **exact integer equality** on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW), counts and flags; euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Every array and every loop bound in a JAX simulator is fixed-size, so there are five hard ceilings. These
scenarios **certify that the documented behaviour at each ceiling is what actually happens** — they
are not fix requests, and that framing must survive into the certificate. The point is the
distinction between a *known, flagged, bounded* degradation and a *silent* one. Overflowing a book
and raising the full flag is an accepted limitation recorded in `book-depth-cap-finding.md`;
overflowing it silently and losing orders would be a defect. Hitting the iteration bound must set the
truncation flag and **still** conserve quantity: the flag may be the only observable consequence,
never a wrong trade list.

This is the **only** scenario permitted to reach a ceiling; every other asserts the opposite through
I8. Together the two halves let the certificate say both "the engine is correct in its operating
envelope" and "its behaviour at the envelope's edge is known, and flagged except where C14a′ records an
unflagged case". Note that these caps have
**no market-spec section at all** — §11.2 is the modeling-choices list and has nothing to do with
array capacities — which is precisely the honest framing: C14 certifies *implementation limits*, and
each is a **departure from** §6.1/§6.2, not an instance of them.

*(Verbatim from `scenario-ladder.md` v2.4 §C14.)* The parts, as the specification states them:

| | Part | Dataset | Known result |
|---|---|---|---|
| **C14a** | Book depth | >100 resting on one side | Full flag raised; worst-price level wiped; no trade manufactured; evicted quantity accounted in its own I2 bucket |
| **C14b** | Trade ring | >100 trades | Documented behaviour: once the ring is saturated, every new trade lands in its **last row**, so a burst keeps only its final trade. The part certifies that this is what happens and that it is detectable, never that it is correct |
| **C14c** | Outer matching passes | Pass count > 100 (`_TRIP_BOUND`; *corrected 2026-10-07, was "Hop count >200": `max_match_iterations` = 200 never binds*) | Loop terminates, truncation flag set, and that flag is the **only** observable — I2 still closes. The iteration count itself cannot be read after the fact, so the part is driven by constructions whose hop count is known analytically. Those constructions keep every iceberg within the slice margin $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$ and assert `truncated == False` after the last registration, before the triggering aggressor, so the flag is attributable to the iteration ceiling and not to C14e's |
| **C14a′** | Regen-slice rests on a full side | Iceberg regeneration into a side already at 100 | Documented limit: a regenerated slice that rests **does not raise** the full flag, so this eviction is unflagged. **Recorded, not certified**: the part observes and reports the unflagged eviction (quantity read from the book diff, rule 2) as a known engineering gap outside the certified surface; it passes if the eviction is exactly the documented one and fails if any other quantity is lost. This is why I8 never relies on the flag alone |
| **C14d** | Iceberg slots | >1024 concurrent icebergs | Bounded per documented behaviour; no side-channel corruption |
| **C14e** | Slices per iceberg | A boundary pair of resting icebergs that never trade, each in a fresh book: $\lceil \mathit{total}/\mathit{peak} \rceil = 65$ and $= 66$ | Documented behaviour (`jax_constants.py` §B2; the capacity-validity precondition): the 65-slice iceberg leaves `truncated == False`, and the 66-slice iceberg sets it **at registration**, with no aggressor and no trade. In both, the book holds exactly the visible peak and I2 closes, with the hidden remainder in its own bucket. What an over-ceiling iceberg does **afterwards** is **recorded, not certified**: its regeneration past the 64th slice, and the fill-or-kill pre-check, which projects only 64 hidden slices. §B2 names this hazard but documents no exact behaviour for it |

**Certifies** (card): no market-spec section. Grounded in `book-depth-cap-finding.md`, OQ-10's 2026-06-11 accepted-limitation ruling, OQ-15 (ring occupancy), OQ-20, the `OMIE-ICE-INCOMING-FILL-REPORT-1` close-out record in `sprint-status.md` (2026-09-28, `1f4755a`) for C14a′, C14b and C14c's observability limits, and `jax_constants.py` §B2 (`MAX_SLICES_PER_REGEN`) for C14e. Each part is a departure from §6.1/§6.2, not an instance of them. **Pass** (card): an exact match against the documented behaviour of each part (C14a′ as a recorded observation); every excluded invariant violated exactly as expected and that violation asserted; every other invariant holds; the eager and jitted runs agree; assertions in integer ticks; I9 holds; every quoted passage still matches its named source document. **Does not certify** (card): correctness at a ceiling; in the truncated regime (C14c only), the post-loop re-submit of a NON order that does not regenerate icebergs, leaving a consumed slot (recorded, not asserted against). **Failure implicates:** a silent capacity overflow, strictly worse than the documented degradation.

**What this notebook asserts, per part** (section 5 states each value as a formula over the draw's parameters; section 7 compares, recording expected and observed **in full** as canonical compact JSON, every list whole). Re-authored at K2 under `OMIE-CERT-RERUN-1` (r2): C14a′ rebuilt, C14c retargeted at the 100-pass bound with a below-bound control, finding F4 corrected, C14c's full outcome recorded, and the two last-row lookup probes added as passing controls (S2, `OMIE-JAXOB-LOOKUP-MISS-1`).

| part | ceiling (configured value) | asserted exactly |
|---|---|---|
| **C14a** | `book_depth` = 100 resting per side | the full side holds 100 orders and `book_full` is still false; the 101st resting add raises `book_full` (sticky); **every** order at the worst price is wiped and lands in the `evicted` bucket at its full quantity; every other order is untouched; no trade; I2 closes with the evicted bucket |
| **C14a′** *(recorded, not certified)* | `book_depth` = 100 | **full only at regeneration** (finding F16): no flag site ever dispatches on a full side. An incoming iceberg trades its peak against the only opposite order and its regenerated slice is the order that brings the side to 100; a later opposite aggressor consumes that slice, so the next regeneration is the only dispatch on a side that held 100 when the call began. Asserted: `book_full` false on every post, the two trades, the slice and the hidden remainder, and the eviction set **exactly as the documented row accounting derives it (empty)**; any lost quantity fails the part; I8 still sees the side at 100 through its resting-per-side axis |
| **C14b** | `nTrades` = 100 trade rows | 100 trades fill the ring exactly; a burst of $n_b$ trades (one aggressor against one resting iceberg) then keeps **only its final trade**, in the ring's last row; rows 0–98 unchanged; the per-submission row usage and the saturation flag pinned; the engine **reports the burst's true quantity** $T_b$ (finding F4, corrected); I2 violated on exactly the burst's two orders by exactly the lost quantity; I6 violated on exactly the burst submission (report $T_b$ against rows $r_b$); I1 passes |
| **C14c** | outer matching passes, `_TRIP_BOUND` = 100 (`max_match_iterations` = 200 is inert) | four resting icebergs (each ≤ 39 slices, $\sum$ slices in $[112, 156]$, so the analytic hop bound $\sum s_j + 3 \le 159$ sits **inside** the generator's 160 hop margin) registered with `truncated` false; one aggressor that needs more than 100 passes raises `truncated`; the loop terminates; I2 closes; every fill is the documented walk up to an unobservable cut point; $0 <$ filled $< Q$; the **full outcome** (`filled`, `still_resting`, `hidden_remaining`, `final_book`, `reported_qty`) recorded and asserted consistent with the observed fills; the final book shows the Rule-4 departure, and I3's violations, if any, sit only on the truncating submission |
| **C14c-below** *(control)* | the same, below the bound | the same construction with each iceberg ≤ 24 slices and $\sum$ slices in $[84, 96]$, so the hop bound $\le 99 < 100$: `truncated` stays false, the walk completes exactly (every slice, in price order, ring rows $0 \ldots \sum s_j - 1$), the book ends empty, every battery check passes |
| **C14d** | `max_icebergs` (configured cap $M$, finding F8) | $M$ icebergs register; the $(M{+}1)$-th sets `truncated` at registration and leaves the side channel byte-identical (no slot overwritten); its peak rests and its hidden remainder is untracked (I2 violated on exactly that order); a later sweep of the best iceberg still trades every slice correctly |
| **C14e** | `MAX_SLICES_PER_REGEN` = 64 | a boundary pair in fresh books: the 65-slice iceberg leaves `truncated` false, the 66-slice iceberg sets it at registration; in both no trade, the book holds exactly the visible peak, and the hidden remainder is in its own bucket |
| **C14-LR-NON** *(control, S2)* | `book_depth` = 100, both sides | both sides hold 100 (each side's last physical row occupied, neither by the order that trades); a NON aggressor on a full own side executes completely against the opposite best: exactly one trade; the full-side wipe fires on that dispatch (`book_full` raised, the own side's worst level evicted, `book-depth-cap-finding.md` correction 1a); **nothing else** is lost — both last-row occupants survive |
| **C14-LR-ICE** *(control, S2)* | `book_depth` = 100 | an iceberg registered first (first physical row), then 99 orders fill its side to 100 (last row occupied by another order); two opposite aggressors consume its peak and its next slice: each regeneration rests at the iceberg's price and the second aggressor reaches it; no flag, nothing evicted, nothing stranded |

**Battery exclusions** (card): I8 on all parts; I1, I6 and the row-derived I2 terms on C14b; I3 on C14c (the truncated construction; its below-bound control runs I3 as usual). Each excluded check is folded into the invariant table as `not_applicable` and its actual result is asserted separately, exactly (section 7); every other check must reach the status section 5 states, and those statuses are asserted too. **Judging rule:** a battery status of `not_applicable` counts as vacuously satisfied wherever `pass` is expected (I9 guards vacuity separately); it is never counted as a departure.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.14**, and `scenario-ladder.md` is **v2.4** (the 100-pass bound, `OMIE-CAPACITY-DOCS-1`). Every quote is checked against the current file at run time.
- **F2 — the dataset is built in the notebook.** The generator has no C14 registry key. Section 4 builds every tape with the generator's public helpers (`TapeBuilder`, `tape_capacity_bounds`, `check_envelope`, `Draw`) from `numpy.random.Generator(PCG64(seed))`, and passes each tape's analytic bounds through `check_envelope(C14_CEILING_OPT_OUT, …)`, the one sanctioned opt-out. The results JSON lists no generator digest (`constructions=()`); section 4 prints each part's digest over its seeds. `draw.expected` is empty and never read.
- **F3 — which grounds are machine-checked.** The checker reads `market-spec.md`, `open-questions.md` and `scenario-ladder-review.md` only. OQ-10, OQ-15, OQ-20, the review's capacity section (with its 2026-10-07 correction note) and its C14 findings, and the market rules each part departs from are checked quotes. The card's other grounds (`book-depth-cap-finding.md` with correction 1a, the `OMIE-ICE-INCOMING-FILL-REPORT-1` close-out in `sprint-status.md`, `jax_constants.py` §B1/§B2, `omie_config.py`, `scenario-ladder.md` v2.4, the `OMIE-JAXOB-LOOKUP-MISS-1` card) are shown as **unchecked citations**. In the release re-run (the maintainer's K7-U verdict REPEAT at K2, 2026-10-27, certification-log entry 15), the unchecked citations that named research documents not part of the release were re-pointed to shipped files that carry the same fact, or dropped where none does.
- **F4 — corrected (r2): the engine reports the true burst quantity.** The first run's F4 derived that the engine's report collapses with the ring; it does not. The engine reports the burst's **true** traded quantity $T_b$, while the record reads the burst's rows by diff and sees only the surviving row ($r_b$). With this battery: I1 still **passes** (it reads the buyer and the seller leg of each fill from the same trade row, so a clobbered row is still balanced); I6(b) **fails on exactly the burst submission**, by the quantity violation (reported $T_b$ against $r_b$ in its rows), and may also carry that submission's settlement-value violation (the documents do not say whether the reported settlement is the true or the row value; section 7 admits it on that submission only, and nothing else); I2 fails on exactly the burst's two orders. The excluded checks' expected results are asserted exactly so.
- **F5 — the C14b burst has one maker.** A burst that walks several resting orders would, on a saturated ring, show only its last fill, against the worst-priced maker, while better-priced makers still show depth: I4 would fail, and the card does not exclude I4. The burst is therefore one aggressor against one resting iceberg (§4.2.1 Rule 2), whose slices are all at one price.
- **F6 — C14c's pass count (r2: the 100-pass bound).** The binding ceiling is the outer loop's `_TRIP_BOUND` = 100 passes; `max_match_iterations` = 200 is configured but never read by the loop (`scenario-ladder.md` v2.4, review §5 correction note). No document says how many slices one pass may consume. The construction puts the four icebergs at **distinct** prices, so §6.1 price priority and §4.2 (a new slice of the best iceberg is activated, at the same best price, before any worse level may trade; `OMIE-ICE-PRICE-PRIORITY-1`) force each pass to stop at an iceberg that still has a slice to regenerate; at most one pass per iceberg boundary can also reach the next iceberg's visible slice. The aggressor therefore needs at least $\sum_j s_j - 3 \ge 109 > 100$ passes and cannot complete; the post-loop re-submit adds at most one fill per iceberg, so $\mathrm{filled} < Q$. The below-bound control has the sound analytic hop bound $\sum_j s_j + 3 \le 99$, so the loop completes before the 100th pass and `truncated` stays false. Both C14c draws sit inside the generator's 160 hop margin: the pair shows that between 100 and 160 only the run's `truncated` check enforces the ceiling, as the capacity precondition states. The ring is configured to `C14C_N_TRADES` (default 256), above the most trades either tape can produce ($\sum s_j + 1$), so neither reaches the trade ceiling.
- **F7 — C14c and I3.** The documents do not say whether a truncated walk leaves a **visibly** crossed book (I3 fails) or a **consumed slot** (an iceberg with hidden quantity and no visible slice opposite the aggressor's resting remainder: the ladder's out-of-scope note; I3 passes). The expectation asserts the disjunction (the Rule-4 departure is present in the final book), that I3 fails exactly when the book is visibly crossed, and that any I3 violation sits only on the truncating submission, so the exclusion cannot hide an earlier failure.
- **F8 — C14d runs at a configured slot cap.** More than 1 024 concurrent icebergs at the default configuration is unreachable without first overflowing the 100-row book (every live iceberg holds a visible peak, at most 2 × 100), which would merge C14d into C14a; and an eager run of 1 025 iceberg submissions does not fit the runtime. The slot cap is a runtime configuration field (`OMIEMarketConfig.max_icebergs`, an axis independent of `book_depth`), and the mechanism (registration on a full side channel) is the same at any value, so C14d runs at `C14D_MAX_ICEBERGS` (default 4). The default 1 024 is shown in section 3.
- **F9 — C14d and I2.** The `jax_constants.py` §B1 docstring says an overflowing registration returns the side channel unchanged and raises `truncated`; the close-out's audit says an overflowed table "reports the peak only". So the overflowing iceberg's peak is dispatched and its hidden remainder is tracked nowhere: I2 fails on exactly that order, by exactly $T - p$. The card does not list I2 on C14d among the exclusions; the expectation treats it as expected-violated and asserts it exactly.
- **F10 — when `book_full` rises.** `book-depth-cap-finding.md` correction 1a: the wipe runs after matching on the incoming order's own side and fires whenever that side is full, even when the order executes completely; `book_full` is raised by NON/IOC passes and by the incoming-iceberg sweep and slice dispatch on a full own side, never by a regeneration. A side holding 99 is not full, so the order that brings it to 100 neither wipes nor flags; the expectation asserts `book_full` false through the 100th order and true from the first flag-site dispatch on a side already at 100.
- **F11 — C14a's overflowing order** is priced strictly better than the worst level (the fate of an overflowing order priced at or behind the worst level is not documented), and the worst-level orders are shuffled among the 100, so "worst price" is never "last in".
- **F12 — I8 on C14e-65.** The battery's I8 has no slice axis, and the 65-slice iceberg raises no flag, so the battery's I8 passes there. The slice margin ($\lceil \mathit{total}/\mathit{peak} \rceil \le 64$) is part of I8 by the capacity-validity precondition and is re-checked from the tape; the expectation asserts the **complete** I8 breach set = the battery's axes and flags plus the slice margin. On every part it is non-empty, and exactly as section 5 states.
- **F13 — C14e boundary.** From the precondition's formula, the flag is set exactly when $\lceil (T-p)/p \rceil > 64$. The pair shares peak, remainder, side, price and identifiers; only one slice differs.
- **F14 — identifiers and incidentals are varied.** The full side alternates BUY/SELL over consecutive seeds (both sides on every part with `DRAWS` ≥ 2); prices, peaks, slice counts, contract id, order ids (1–999 999), agent ids and the first tape timestamp are drawn on every seed.
- **F15 — what is recorded, not asserted.** C14d's live-slot count after the sweep (cleanup timing undocumented); C14c's cut point and trade count (reported in full; its outcome is asserted only as consistent with its own fills); what an over-ceiling iceberg does after registration (C14e has no aggressor, so it is not exercised).
- **F16 — C14a′ rebuilt: full only at regeneration; the documented eviction is not derivable.** The first run's C14a′ filled the side to 100 **before** the incoming iceberg, so the iceberg's own sweep was a dispatch on a full side: by correction 1a that dispatch wipes and raises `book_full`, so the eviction it observed was flagged and was not the C14a′ case. The rebuild keeps every flag site off a full side: the side holds 99 when the iceberg arrives, its regenerated slice is what brings the side to 100, and the next regeneration (after an opposite aggressor consumes that slice) is the only dispatch on the side. **Finding:** the documents also imply that a consumed slice leaves the book before its iceberg regenerates (OQ-20: the consumed peak's id is absent from the book; `OMIE-JAXOB-LOOKUP-MISS-1`: the stranding was a *miss* on exactly that lookup). A regeneration therefore replaces the row its own consumed slice vacated and meets at most 99 other orders, so no flag-free construction derivable from the documents reaches "a regenerated slice resting on a side already at 100"; the documents give a consumed slot that survives an arrival only through a flagged wipe or the truncated regime. The expectation states the derived result (no eviction, no flag); an eviction here would be the documented unflagged wipe and is reported as a recorded departure, never hidden.
- **F17 — the last-row lookup controls (S2).** `OMIE-JAXOB-LOOKUP-MISS-1` (merged) fixed the jaxob lookup that returned a side's last physical row on a miss, which silently deleted the order resting there (crossing NON) and stranded an iceberg. The first run's probes become passing controls here, rebuilt from the card's trigger (100 resting orders on a side, last row occupied): C14-LR-NON and C14-LR-ICE. Rows fill first-free in submission order, so the order added 100th to a side sits in its last row; it is never the order that trades nor at the worst level, so its survival is informative.

## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. **No market-spec section certifies C14** (card): the market-spec passages below are the rules each part **departs from**, quoted so the departure can be named. The engineering grounds the card cites from documents the checker cannot read are shown after them as **citations**, rendered but not machine-checked (finding F3).

How the quotes ground the expectation of section 5:

- **The accepted limitation (OQ-10, `book-depth-cap-finding.md`, review §5).** A full side's worst-price level is wiped: an accepted, documented limitation, flagged by `book_full`. Hence C14a: every worst-price order evicted, at its full quantity, into its own I2 bucket; nothing else lost.
- **The wipe trigger (`book-depth-cap-finding.md` correction 1a, cited).** The wipe runs after matching on the incoming order's own side and fires on any dispatch against a full own side, even one that executes completely; regenerations are never flag sites. Hence F10, C14-LR-NON's flagged wipe, and the rebuilt C14a′ (F16).
- **The unflagged eviction (close-out record, ladder engineering limits, review V3-14).** A regenerated slice that rests on a full side evicts without `book_full`. Hence C14a′, recorded, not certified: rebuilt so the side is full only at regeneration, and the eviction must be exactly the one the documented row accounting derives (F16).
- **The ring (OQ-15, review §5 item 2 and §11.1).** On a saturated ring every further trade is written to the last row, and the diff-based fill count collapses to one. Hence C14b: a burst keeps only its final trade, the engine still reports the true quantity (finding F4, corrected), so I6 and I2 lose exactly the clobbered quantity.
- **The pass bound (§4.2.1 Rules 2 and 4, §4.2, §6.1, review §5 item 1 and its 2026-10-07 correction note, V3-15).** Multi-slice matching runs inside one call, and the cascade is paid out of the 100-pass budget (`_TRIP_BOUND`; `max_match_iterations` = 200 never binds); truncation is a Rule-4 departure. Hence C14c and its below-bound control (findings F6, F7).
- **The last-row lookup (`OMIE-JAXOB-LOOKUP-MISS-1`, cited; §6.1, I2).** A lookup miss is a miss, so a side whose last row is occupied loses nothing it should not and its icebergs regenerate. Hence the two S2 controls (F17).
- **The side channel (OQ-20, §4.2.1 Rule 4 scope clarification, `jax_constants.py` §B1).** A slot whose visible peak is consumed is a permitted between-aggressor state; an overflowing registration leaves the side channel unchanged and raises `truncated`. Hence C14d.
- **The slice ceiling (`jax_constants.py` §B2, the capacity-validity precondition).** `truncated` is raised at registration exactly when $\lceil (T-p)/p \rceil > 64$. Hence C14e. The ICE row of §4.5 is why the hidden remainder rests in its own bucket; its NON row is why a truncated aggressor's residual rests (C14c).
- **The exclusions (review V3-15).** Each excluded invariant is reported as expected-violated and its violation asserted.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- C14a: the accepted limitation (book depth) ---------------------------------------------------
    SpecQuote(
        'open-questions.md',
        'OQ-10',
        'Book-overflow eviction (added 2026-06-11, `P3-PREP-REPLAY-1-FOLLOWUP-1`): the full-side worst-price level-wipe at the `book_depth=100` cap is an **accepted documented limitation**',
        block='**Decision impact:**',
        note='C14a: the worst-price level wipe is the documented behaviour at the book-depth ceiling, a limitation and not a fix request.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5',
        "The plan's principle — every scenario except the capacity one must stay strictly inside the ceilings, or its result does not count — is domain-correct, and the framing of the caps as *documented degradations* rather than defects matches OQ-10's accepted-limitation ruling.",
        note='Why C14 is the only scenario allowed to reach a ceiling, and why its oracle is the documented degradation.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5 item 3',
        'the documented overflow is a full-side *worst-price level wipe*',
        note='C14a: every order at the worst price of the full side is evicted, not one order.',
    ),
    # --- C14a′: the unflagged eviction (recorded, not certified) --------------------------------------
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-14',
        '**Recorded, not certified**: the part observes and reports the unflagged eviction (quantity read from the book diff, rule 2) as a known engineering gap outside the certified surface; it passes if the eviction is exactly the documented one and fails if any other quantity is lost',
        note='C14a′: the eviction must be exactly the worst-price level; any other lost quantity fails the part.',
    ),
    # --- C14b: the trade ring --------------------------------------------------------------------------
    SpecQuote(
        'open-questions.md',
        'OQ-15',
        'On a saturated buffer no slot equals `-1`, so every subsequent `add_trade` call writes to index `-1` (Python-style last-slot), silently clobbering earlier fills of the same `submit_order` invocation.',
        block='**What is unknown (engineering invariant):**',
        note='C14b: once the ring is full, every further trade is written to its last row, so a burst keeps only its final trade.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-15',
        'Under multi-level crossings on a saturated buffer, the diff-based fill count collapses to `num_fills == 1`.',
        block='**Consequence for Phase 2 (observed):**',
        note="C14b: the record's row diff collapses the burst to its one surviving row; the engine's own report "
             "does not collapse (finding F4, corrected r2), so I6 sees the difference.",
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§11.1',
        '"Every further trade lands in the last row" is OQ-15\'s documented mechanism',
        note='The C14b known result, confirmed by the domain review.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5 item 2',
        'A multi-level walk, an iceberg cascade, a settlement draw or a fuzz session can each exceed 100 trades, and a wrap silently corrupts I1/I2/I6 — *the very invariants used as the oracle*.',
        note='Why the ring is excluded from every other scenario; on C14b the corruption is asserted exactly (findings F4, F5).',
    ),
    # --- C14c: the iteration bound and Rule 4 ----------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'A single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle.',
        note='Rule 2: one aggressor consumes slice after slice inside one call (C14b burst, C14c, C14d sweep).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**the entire cascade MUST resolve to a fixed point within the originating matching process** (all resultant trades carry that matching process's timestamp; the order book is never left in a crossed state between aggressor arrivals)",
        note='Rule 4, the rule a truncated walk departs from (C14c): the I3 exclusion and the Rule-4 departure asserted in section 7 (finding F7).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'the order book "is updated whenever orders are added to it, deleted from it, modified (volume, price, execution restriction, **activation of a new iceberg slice**)... **After each order book update... the algorithm validates if new matching opportunities have arisen and, if so, the resulting trades are concluded.**"',
        note='Each slice activation is a book update followed by a matching step: why each activation costs the walk at least one iteration (finding F6).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5 item 1',
        'Rule 4 requires the cascade to reach a fixed point *within the originating matching process*, so the iteration budget is the resource conformance is paid out of.',
        note='C14c: the pass budget pays for the regeneration cascade; past it, the walk is truncated.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5',
        'outer match loop runs a fixed `_TRIP_BOUND` = 100 passes and does not read `max_match_iterations`;',
        note='C14c (r2): the binding ceiling is 100 outer passes, not the configured 200 iterations (finding F6).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5',
        '`truncated` fires on either bound, so 200 can never bind first.',
        note='C14c (r2): the draw is built between 100 and 160 hops, so only the 100-pass bound can raise the flag.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§5',
        'A binding bound is flagged (`truncated`), never silent.',
        note='C14c: truncated raised on the truncating submission; C14c-below: never raised.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-15',
        'A truncated cascade can leave a regenerated slice resting in a crossed book. That breaks I3 and is a §4.2.1 Rule 4 departure by construction',
        note='C14c: the I3 exclusion (finding F7 states which shapes the departure may take).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.',
        note='Slices are activated one at a time, at the iceberg price: every slice query below uses ceil(T/p) slices of p, the last one T - (n-1)p.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Buy side:** ranked highest price first; ties broken by earliest timestamp.',
        note='The worst bid level is the lowest price; a SELL aggressor walks bids from the highest.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Sell side:** ranked lowest price first; ties broken by earliest timestamp.',
        note='The worst ask level is the highest price; a BUY aggressor walks asks from the lowest, so C14c consumes its icebergs one price level after another.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note='Every pinned trade prints at the resting order\'s (or the iceberg\'s, §6.2 item 7) price.',
    ),
    # --- C14c/C14d: the side channel -------------------------------------------------------------------
    SpecQuote(
        'open-questions.md',
        'OQ-20',
        'It does not impose a post-call invariant on simulator-internal side-channel state (`iceberg_meta`).',
        block='**Ruling.**',
        note='C14c and C14d assert the observable book and the I2 buckets, never a side-channel slot state, except where §B1 documents one (C14d).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'A side-channel slot whose `order_id` is absent from the LOB is consistent with the spec when the slot represents an iceberg whose previous peak was fully consumed and whose next peak has not yet been activated.',
        note='The consumed-slot shape C14c may leave (the ladder\'s out-of-scope note), which I3 cannot see (finding F7).',
    ),
    # --- C14e: hidden quantity rests; the quantity bound --------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |',
        note='C14e: the hidden remainder rests, in its own I2 bucket, whether or not the iceberg is over the slice ceiling.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| NON | Yes | Yes | Regular, Block |',
        note="C14c: a NON aggressor's unfilled residual rests at its limit (the post-truncation outcome).",
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).',
        note='Every drawn quantity, iceberg totals included, stays within 1 to 9 990 ticks.',
    ),
    # --- the exclusions ------------------------------------------------------------------------------------
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-15',
        'Under OQ-15 a saturated ring clobbers rows (`open-questions.md:422`). That breaks I1 and I6, which are read from trade rows, and any I2 term derived from rows.',
        note='The C14b exclusions. Finding F4 (corrected r2): the clobbering reaches I2 and I6 (exactly the burst), and I1 '
             'is asserted to pass.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-15',
        'Each excluded invariant is reported as **expected-violated**, and its violation is itself asserted, so the exclusion cannot hide an unrelated failure.',
        note='Section 7 asserts every excluded check\'s actual result exactly.',
    ),
)

CITATIONS = (
    Citation(
        'docs/omiecid/open-questions.md:271, OQ-10 **Decision impact:** (the book-depth cap finding\'s status and '
        'its level wipe)',
        'the full-side worst-price level-wipe at the `book_depth=100` cap is an **accepted documented limitation** […] '
        '(status now "Closed — documented accepted limitation (2026-06-11)")',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md v2.5, §The capacity-validity precondition, book-depth row and "The pass '
        'ceiling" (the book-depth cap finding\'s correction 1a, 2026-10-07, OMIE-CAPACITY-DOCS-1)',
        'A regenerated iceberg slice that rests on a full side evicts **without** raising the flag […] '
        'The book-depth row\'s wipe also fires on a dispatch that executes completely against a full own side',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md v2.5, §The capacity-validity precondition, "The pass ceiling"',
        'That value does not bound the outer match loop: the loop\'s continue test does not read it, and the loop '
        'runs a fixed `_TRIP_BOUND` = 100 passes (`contract_book.py`). The `truncated` flag fires when either bound '
        'is reached with quantity left, so only the 100-pass bound can ever bind',
    ),
    Citation(
        'omiecid_simulator/jaxob/JaxOrderBookArrays.py:1050-1074, get_order_by_id (the lookup-miss contract restored '
        'by OMIE-JAXOB-LOOKUP-MISS-1, certification fix pass S2)',
        'Returns an empty array (-1 dummy […] miss test below is a traced scalar comparison (jit-safe lookup miss). […] '
        '# return vector of NEGATIVE_RETURN_ID if not found',
    ),
    Citation(
        'omiecid_simulator/jaxomiecid/omie_data_loader.py:129-131, ReplayResult (the book_full flag)',
        'the terminal :class:`OMIEBookState` (the ``lax.scan`` carry), […] ``book_full`` full-side-eviction '
        'diagnostic.',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md §C14, part C14a′ (the OMIE-ICE-INCOMING-FILL-REPORT-1 close-out finding F1)',
        'Documented limit: a regenerated slice that rests **does not raise** the full flag, so this eviction is '
        'unflagged.',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md §Out of scope, engineering limits (the OMIE-ICE-INCOMING-FILL-REPORT-1 '
        'deferrals)',
        'when the iceberg fill-report defect was fixed, none a market rule […] '
        'The order-book tape\'s empty-slot detector reads the right value for the wrong reason',
    ),
    Citation(
        'omiecid_simulator/omiecid/jax_constants.py:75-81 (§B1, slot overflow)',
        '``IcebergManager.register`` returns ``(updated_meta, overflow_bool)``; when the meta is saturated at '
        'registration time, ``overflow_bool`` is True and the meta is returned unchanged (no slot is silently '
        'overwritten). The overflow flag propagates through ``ContractBook._build_state`` onto the sticky '
        '``OMIEBookState.truncated`` field',
    ),
    Citation(
        'omiecid_simulator/omiecid/jax_constants.py:112-116 (§B2, MAX_SLICES_PER_REGEN = 64)',
        'HAZARD — truncation-silent bound: if a session ever produces an iceberg with ``remaining_hidden / '
        'peak_qty > MAX_SLICES_PER_REGEN``, the ``jax.lax.while_loop`` terminates early and the remaining hidden '
        'volume is silently dropped from matching.',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md §The capacity-validity precondition, "The slice ceiling" (v2.2)',
        'At registration no slice has been issued and $h = \\mathit{total} - \\mathit{peak}$, so the flag is set '
        'exactly when $\\lceil (\\mathit{total}-\\mathit{peak})/\\mathit{peak} \\rceil > 64$, that is when '
        '$\\lceil \\mathit{total}/\\mathit{peak} \\rceil \\ge 66$.',
    ),
    Citation(
        'docs/omiecid/scenario-ladder.md §Out of scope, engineering limits (C14a′, C14c)',
        'A regenerated iceberg slice that rests on a full side does **not** raise `book_full` (C14a′). […] In the '
        'truncated regime (C14c only), the post-loop re-submit of a NON order does not regenerate icebergs, leaving '
        'a consumed slot.',
    ),
    Citation(
        'omiecid_simulator/omiecid/omie_config.py:35-37, :47 (the configured ceilings)',
        'book_depth: int = 100       # max resting orders per side per contract […] nTrades: int = 100          # '
        'trades array capacity for jaxob (per matching call) […] max_match_iterations: int = 200  # truncation cap; inert '
        'while > `_TRIP_BOUND` (100), which is the effective outer pass bound […] max_icebergs: int = 1024',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 3`** draws **per part**. C14 is exact-parametric on deterministic constructions, so each draw varies the incidentals (side, prices, sizes, identifiers, F14) rather than sampling a distribution. A *draw* is one part on one seed; part $j$ uses seeds `SEED0 + PART_SEED_BASE[j] + k`, $k = 0 \ldots$ `DRAWS`$-1$, so every draw has its own seed. The first seven parts keep the first run's seed bases (C14c is still `C14_seed300000`); the three r2 parts (`c_below`, `lr_non`, `lr_ice`) are appended after them. The C14e pair shares its parameters: part `e66` draws them from `e65`'s seed (finding F13). Requested draws $= 10 \times$ `DRAWS`. C14 has **no discards**: every construction is built to reach its ceiling (or, for a control, to sit exactly where it says), I8 never invalidates a C14 draw (it is excluded and asserted), and a run that raises is a failed check (and, for the accounting, a discard with its reason).
- **`EAGER_PER_PART = 1`.** The first `EAGER_PER_PART` **valid** draws of every part (in seed order: the dataset precondition holds and the jitted run returned a record) also run eager, compared with the jitted run on every integer field (`compare_records`). A part with fewer eager comparisons is a failed `eager_jit` assertion.
- **`C14C_N_TRADES = 256`**: the trade-ring size C14c and its below-bound control run with (finding F6). It must exceed the most trades either tape can produce, $\sum s_j + 1$ (each trade exhausts at least one unit); section 4 checks it per draw.
- **`C14D_MAX_ICEBERGS = 4`**: the slot cap C14d runs with (finding F8). Every other part runs the default `OMIEMarketConfig()`.
- **The ceilings.** The five configured values are read from the configuration dataclasses and `jax_constants.py`, never typed in; section 7 asserts they are 100, 100, 200, 1 024 and 64. The **binding** pass ceiling is not a configuration field: it is the outer loop's fixed `_TRIP_BOUND` = 100 (`scenario-ladder.md` v2.4), stated here as `PASS_BOUND`, and the configured `max_match_iterations` = 200 is inert. The capacity-validity precondition's five ceilings are therefore 100, 100, **100**, 1 024 and 64.

In [ ]:
CERT_ID = "C14"            # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 3                  # draws per part (10 parts: a, a_prime, b, c, d, e65, e66, c_below, lr_non, lr_ice)
SEED0 = 0                  # first seed (part j uses SEED0 + PART_SEED_BASE[j] + k)
EAGER_PER_PART = 1         # the first EAGER_PER_PART valid draws of every part also run eager
C14C_N_TRADES = 256        # C14c / C14c-below trade-ring size: above the most trades either tape can produce (F6)
C14D_MAX_ICEBERGS = 4      # C14d's configured iceberg-slot cap (finding F8)

In [ ]:
import dataclasses
import json
import pathlib
import types

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.jax_constants import MAX_SLICES_PER_REGEN
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig, OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

CONSTRUCTION = "C14.capacity_edges (built in section 4)"   # not a generator registry key (finding F2)
CONFIG = OMIEProductConfig()          # builds tapes and decodes for display; never a source of the expectation
ENVELOPE = synth.DEFAULT_ENVELOPE     # the I8 margins every OTHER scenario keeps (80 / 80 / 160 / 800)
MARKET = OMIEMarketConfig()           # the default configuration: the ceilings C14 certifies

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)

# ---- the five ceilings, read from the configuration (never typed in) --------------------------------------
CEILINGS = {
    "book_depth (resting orders per side)": int(MARKET.book_depth),
    "nTrades (trade rows per contract)": int(MARKET.nTrades),
    "max_match_iterations (per submission)": int(MARKET.max_match_iterations),
    "max_icebergs (concurrent icebergs)": int(MARKET.max_icebergs),
    "MAX_SLICES_PER_REGEN (slices per iceberg)": int(MAX_SLICES_PER_REGEN),
}
CONFIGURED_CEILINGS = [100, 100, 200, 1024, 64]  # the five configured values (max_match_iterations = 200 is inert)
PASS_BOUND = 100                                  # _TRIP_BOUND: the binding outer-pass ceiling (scenario-ladder v2.4)
DOCUMENTED_CEILINGS = [100, 100, PASS_BOUND, 1024, 64]   # the capacity-validity precondition's five ceilings (v2.4)
DEPTH = int(MARKET.book_depth)
RING = int(MARKET.nTrades)
ITER = int(MARKET.max_match_iterations)
SLICE_CEIL = int(MAX_SLICES_PER_REGEN)
SLICE_MARGIN = SLICE_CEIL                         # generator margin: ceil(total / peak) <= 64

# ---- the parts ---------------------------------------------------------------------------------------------
# The first seven keep the first run's seed bases (C14c = C14_seed300000); the r2 parts are appended.
PARTS = ("a", "a_prime", "b", "c", "d", "e65", "e66", "c_below", "lr_non", "lr_ice")
LABEL = {"a": "C14a", "a_prime": "C14a′", "b": "C14b", "c": "C14c", "d": "C14d", "e65": "C14e-65",
         "e66": "C14e-66", "c_below": "C14c-below", "lr_non": "C14-LR-NON", "lr_ice": "C14-LR-ICE"}
CEILING_OF = {"a": "book_depth", "a_prime": "book_depth", "b": "nTrades", "c": "outer passes (_TRIP_BOUND)",
              "d": "max_icebergs", "e65": "MAX_SLICES_PER_REGEN", "e66": "MAX_SLICES_PER_REGEN",
              "c_below": "outer passes (_TRIP_BOUND)", "lr_non": "book_depth", "lr_ice": "book_depth"}
PART_SEED_BASE = {p: 100_000 * j for j, p in enumerate(PARTS)}
PART_MARKET = {p: MARKET for p in PARTS}
PART_MARKET["c"] = OMIEMarketConfig(nTrades=int(C14C_N_TRADES))
PART_MARKET["c_below"] = OMIEMarketConfig(nTrades=int(C14C_N_TRADES))
PART_MARKET["d"] = OMIEMarketConfig(max_icebergs=int(C14D_MAX_ICEBERGS))
SEEDS = {p: [SEED0 + PART_SEED_BASE[p] + k for k in range(DRAWS)] for p in PARTS}
REQUESTED = DRAWS * len(PARTS)

# ---- display units -------------------------------------------------------------------------------------------
O_TICKS = 999_900                     # price_int of 0.00 EUR/MWh (§2.3 floor -9,999 at 0.01): display only
PRICE_INT_MAX, QTY_INT_MAX = 1_999_800, 9_990   # §2.3 bounds, in ticks

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C14_capacity_edges.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}; {DRAWS} draws per part x {len(PARTS)} parts = {REQUESTED} requested draws")
print(f"eager: the first {EAGER_PER_PART} draw(s) of every part")
print("the five ceilings (configuration):")
for _k, _v in CEILINGS.items():
    print(f"  {_k}: {_v}")
print(f"binding outer-pass ceiling (_TRIP_BOUND, documented): {PASS_BOUND}; documented ceilings {DOCUMENTED_CEILINGS}")
print(f"I8 margins every other scenario keeps: {ENVELOPE}; slice margin <= {SLICE_MARGIN}")
for _p in PARTS:
    _m = PART_MARKET[_p]
    _diff = {f.name: getattr(_m, f.name) for f in dataclasses.fields(_m) if getattr(_m, f.name) != getattr(MARKET, f.name)}
    print(f"  {LABEL[_p]:10s} ceiling {CEILING_OF[_p]:26s} seeds {SEEDS[_p][0]}..{SEEDS[_p][-1]}  "
          f"market config: {'default' if not _diff else _diff}")

## 4. Dataset

`make_draw(part, seed)` below builds one part's tape from `numpy.random.Generator(PCG64(seed))` with the generator's public helpers and returns a generator `Draw` (finding F2). Every tape is one contract (id drawn in 0–95), one fresh book; order ids are distinct random integers in 1–999 999, agent ids random in 1–400, tape timestamps consecutive from a random start in 1–20; every row is GFS with no expiry and no price increment (ICE `ppd = 0`). $S$ is the **full side** (or the iceberg side): BUY on even seeds, SELL on odd ones, so every part covers both sides; $O$ is the other side. "Better" means more aggressive for its side (a higher bid, a lower ask); "worse" the opposite. Prices are drawn around a mid of 5–300 EUR/MWh; every price and quantity stays inside §2.3 (checked). An iceberg of $n$ slices and peak $p$ has total $T = (n-1)\,p + r$, $1 \le r \le p$, so $\lceil T/p \rceil = n$ by construction; its slices are $p, \ldots, p, r$.

| part | submissions, in order | drawn |
|---|---|---|
| **C14a** | (0) one order on $O$, priced beyond $S$'s best (never crosses); (1–100) **100** NON orders on $S$, in random order: $k$ at the worst price $W$, $100-k$ at prices 1…$L$ ticks better; (101) the **overflowing** NON order on $S$, 1…$L$ ticks better than $W$ | $k \in [1,4]$, $L \in [3,30]$, quantities 1–50 ticks |
| **C14a′** *(r2: full only at regeneration)* | (0–98) **99** NON orders on $S$ as in C14a ($k$ at $W$); (99) **B**: one NON order on $O$ at $b$, priced beyond $S$'s best, of quantity $p$; (100) **x**: an incoming ICE on $S$ at $b$, peak $p$, $n$ slices (it trades its peak against B; its regenerated slice brings $S$ to 100); (101) **A**: one NON order on $O$ at $b$ for exactly $p$ (it consumes x's slice; x's next regeneration is the only dispatch on $S$, which held 100) | $k \in [1,4]$, $L \in [3,30]$, $p \in [1,30]$, $n \in [3,10]$ |
| **C14b** | $K$ pairs (ICE$_i$ resting on $S$ at $P_i$, $n_i$ slices; then a NON aggressor on $O$ at $P_i$ for exactly $T_i$), with $\sum_i n_i = 100$ exactly; then the **burst** pair (ICE$_b$ at $P_b$, $n_b$ slices; its aggressor for exactly $T_b$). Each $P_{i+1}$ is 1–20 ticks better for $S$ than $P_i$, so no aggressor can reach an earlier iceberg | $K \in [2,4]$, $n_i, n_b \in [2,64]$, $p \in [1,15]$ |
| **C14c** *(r2: the 100-pass bound)* | (0–3) four ICE resting on $S$ at distinct prices $P_1$ (best) … $P_4$ (each 1–20 ticks worse), $s_j$ slices each; (4) one NON aggressor on $O$ at $P_4$ for $Q = \sum_j T_j$ | $s_j \in [28,39]$ (so $\sum s_j \in [112,156]$: more than 100 passes needed, hop bound $\sum s_j + 3 \le 159$ inside the 160 margin), $p_j \in [1,30]$; ring `C14C_N_TRADES` |
| **C14c-below** *(control)* | as C14c | $s_j \in [21,24]$ (so $\sum s_j \in [84,96]$, hop bound $\le 99 < 100$), $p_j \in [1,30]$; ring `C14C_N_TRADES` |
| **C14d** | (0…$M$) $M{+}1$ ICE resting on $S$ at distinct prices $P_1$ (best) … $P_{M+1}$; ($M{+}1$) one NON aggressor on $O$ at $P_1$ for exactly $T_1$ | $M$ = `C14D_MAX_ICEBERGS`, $n_j \in [2,8]$, $p_j \in [1,30]$ |
| **C14e-65 / C14e-66** | (0) one ICE resting on $S$ at $P$, alone in a fresh book, $n$ slices: $n = 65$ and $n = 66$ ($= \text{MAX\_SLICES\_PER\_REGEN} + 1, + 2$), same $p$, $r$, $S$, $P$ and identifiers | $p \in [1,150]$ (so $T \le$ 9 990 ticks) |
| **C14-LR-NON** *(control, S2)* | (0–99) **100** NON orders on $S$ as in C14a ($k$ at $W$, shuffled), except that the 100th one (the side's last row) is priced better than $W$; (100–199) **100** NON orders on $O$, all beyond $S$'s best: first the single best one, **E**, at $e$ for $q_E$, then 99 at 1…$L$ ticks worse than $e$ (the 100th, $O$'s last row, is never E); (200) **X**: one NON order on $S$ at $e$ for exactly $q_E$ | $k \in [1,4]$, $L \in [3,30]$, quantities 1–50 ticks |
| **C14-LR-ICE** *(control, S2)* | (0) **Z**: ICE on $S$ at $b$, peak $p$, $n$ slices, into the empty book (first row); (1–99) **99** NON orders on $S$ at 1…$L$ ticks worse than $b$ (the 99th, the side's last row, is a NON order); (100) **A1**: NON on $O$ at $b$ for $p$; (101) **A2**: NON on $O$ at $b$ for Z's second slice $\min(p, T-p)$ | $L \in [3,30]$, $p \in [1,30]$, $n \in [3,10]$ |

Each tape's analytic bounds (`tape_capacity_bounds`) go through `check_envelope(C14_CEILING_OPT_OUT, …)`, the one sanctioned opt-out, and are then compared with the I8 envelope **on purpose**: the part's analytic breach set must be exactly the one stated here (C14a, C14a′, C14-LR-NON, C14-LR-ICE: resting per side; C14b: trades; C14c and C14c-below: trades only — their hop bounds, 115–159 and 87–99, are **inside** the 160 hop margin, which is the point of the r2 retarget; C14d, C14e: none on the four envelope axes — C14d breaches a configured cap and C14e the slice margin, which `tape_capacity_bounds` does not carry). The **dataset precondition** `precondition(draw)` re-derives every structural claim of this table from the tape alone (counts, prices, slice counts, the crossing pattern, the hop bound against `PASS_BOUND`, §2.3 bounds, distinct ids, the ring headroom of C14c, the last-row occupants) and returns every departure; section 7 asserts it is empty for every draw.

Below: each part's first draw, summarised in readable units beside the integers (C14a/C14a′/C14-LR-NON/C14-LR-ICE by price level, the others row by row), every draw's parameters, the per-part construction digests, and the book shape of each part's first draw.

In [ ]:
import hashlib
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_SIDE, C_PRICE, C_QTY, C_OID = synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY, synth.COL_OID
C_AGENT, C_TS, C_EXEC, C_PEAK, C_PPD = synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_PEAK, synth.COL_PPD
C_VALIDITY, C_EXPIRY, C_CONTRACT = synth.COL_VALIDITY, synth.COL_EXPIRY, synth.COL_CONTRACT

MID = (1_000_400, 1_029_900)          # price_int: 5.00 - 300.00 EUR/MWh (empirical test range, never a bound)
AGENT_IDS = (1, 400)
ORDER_IDS = (1, 999_999)
ORDER_KEYS = ("oid", "side", "exec", "price_int", "qty_int", "peak_int", "agent")


def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def rng_for(seed):
    return np.random.Generator(np.random.PCG64(int(seed)))


def ceil_div(a, b):
    return -(-int(a) // int(b))


def side_of(seed):
    # The full (or iceberg) side alternates BUY / SELL over a part's consecutive seeds (both sides always covered).
    return BUY if int(seed) % 2 == 0 else SELL


def opp(side):
    return SELL if int(side) == BUY else BUY


def better(side, price, d):
    # d ticks more aggressive for `side` (a higher bid, a lower ask).
    return int(price) + int(d) if int(side) == BUY else int(price) - int(d)


def worse(side, price, d):
    return int(price) - int(d) if int(side) == BUY else int(price) + int(d)


def is_better(side, a, b):
    # price a strictly better than price b for `side` (§6.1).
    return int(a) > int(b) if int(side) == BUY else int(a) < int(b)


def crosses(buy_price, sell_price):
    return int(sell_price) <= int(buy_price)     # §6.2 item 2


def slices_of(total, peak):
    # §4.2: ceil(T/p) slices of p, the last one T - (n-1)p.
    n = ceil_div(total, peak)
    return [int(peak)] * (n - 1) + [int(total) - (n - 1) * int(peak)]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def eur(p_int):
    return float(decode_price(int(p_int), CONFIG))   # display only


def mw(q_int):
    return float(decode_qty(int(q_int), CONFIG))     # display only


def distinct_ids(rng, n, lo, hi):
    out, seen = [], set()
    while len(out) < n:
        x = int(rng.integers(lo, hi + 1))
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def ice_total(rng, n, p):
    # T = (n-1) p + r, 1 <= r <= p, so ceil(T/p) = n by construction.
    return (int(n) - 1) * int(p) + int(rng.integers(1, int(p) + 1))


class Rows:
    """Accumulates one tape's rows through synth.TapeBuilder, with the declaration of each row."""

    def __init__(self, rng, n_rows):
        self.rng = rng
        self.oids = distinct_ids(rng, n_rows, *ORDER_IDS)
        self.agents = [int(a) for a in rng.integers(AGENT_IDS[0], AGENT_IDS[1] + 1, size=n_rows)]
        self.contract_id = int(rng.integers(0, 96))
        self.tb = synth.TapeBuilder(CONFIG, contract_id=self.contract_id,
                                    first_timestamp=int(rng.integers(1, 21)), timestamp_step=1)
        self.orders = []

    def add(self, side, exec_, price, qty, peak=0):
        i = len(self.orders)
        oid, agent = self.oids[i], self.agents[i]
        self.tb.add(side=int(side), price_int=int(price), qty_int=int(qty), agent_id=agent,
                    exec_restriction=int(exec_), peak_int=int(peak) if int(exec_) == ICE else 0, ppd_int=0,
                    order_id=oid)
        self.orders.append([oid, int(side), int(exec_), int(price), int(qty), int(peak) if int(exec_) == ICE else 0,
                            agent])
        return oid


def finish(part, seed, rows, params):
    tape = rows.tb.build(pad_extra=8)
    cb = synth.tape_capacity_bounds(tape)
    synth.check_envelope(synth.C14_CEILING_OPT_OUT, **cb)    # the one sanctioned opt-out from the I8 envelope
    m = PART_MARKET[part]
    params = {"part": part, "contract_id": rows.contract_id, "orders": rows.orders,
              "market": {"book_depth": int(m.book_depth), "nTrades": int(m.nTrades),
                         "max_match_iterations": int(m.max_match_iterations), "max_icebergs": int(m.max_icebergs)},
              **params}
    return synth.Draw(construction=f"C14.{part}", seed=int(seed), tape=tape, params=params, expected={},
                      hop_bound=int(cb["hops"]), capacity_bounds=types.MappingProxyType({k: int(v) for k, v in cb.items()}),
                      oracle_kind="observation" if part == "a_prime" else "exact-parametric", meta={"part": part})


def _full_side(rng, rows, S, W, k, L, n=DEPTH, last_not_worst=False):
    # n NON orders on S, in random order: k at the worst price W, the rest 1..L ticks better.
    # last_not_worst: the n-th one added (the side's last physical row when n = 100) is never at W.
    prices = [W] * k + [better(S, W, int(rng.integers(1, L + 1))) for _ in range(n - k)]
    qtys = [int(q) for q in rng.integers(1, 51, size=n)]
    order = [int(i) for i in rng.permutation(n)]
    if last_not_worst and prices[order[-1]] == W:
        j = next(j for j in range(n - 1) if prices[order[j]] != W)
        order[j], order[-1] = order[-1], order[j]
    worst, oids = [], []
    for i in order:
        oid = rows.add(S, NON, prices[i], qtys[i])
        oids.append(oid)
        if prices[i] == W:
            worst.append(oid)
    return worst, oids


def build_a(seed):
    rng = rng_for(seed)
    S = side_of(seed)
    k, L, g = int(rng.integers(1, 5)), int(rng.integers(3, 31)), int(rng.integers(1, 51))
    W = int(rng.integers(*MID))
    rows = Rows(rng, DEPTH + 2)
    o_opp = rows.add(opp(S), NON, better(S, W, L + g), int(rng.integers(1, 51)))
    worst, _ = _full_side(rng, rows, S, W, k, L)
    o_over = rows.add(S, NON, better(S, W, int(rng.integers(1, L + 1))), int(rng.integers(1, 51)))
    return finish("a", seed, rows, {"S": S, "W": W, "k": k, "L": L, "worst_oids": worst, "opp_oid": o_opp,
                                    "overflow_oid": o_over, "full_at_submission": DEPTH,
                                    "overflow_submission": DEPTH + 1})


def build_a_prime(seed):
    # r2: full only at regeneration (finding F16). S holds 99 when x arrives; x's regenerated slice brings it
    # to 100; A then consumes that slice, so x's next regeneration is the only dispatch on a side at 100.
    rng = rng_for(seed)
    S = side_of(seed)
    k, L, g = int(rng.integers(1, 5)), int(rng.integers(3, 31)), int(rng.integers(1, 51))
    W = int(rng.integers(*MID))
    p, n = int(rng.integers(1, 31)), int(rng.integers(3, 11))
    T = ice_total(rng, n, p)
    rows = Rows(rng, DEPTH + 2)
    worst, _ = _full_side(rng, rows, S, W, k, L, n=DEPTH - 1)
    b = better(S, W, L + g)
    o_B = rows.add(opp(S), NON, b, p)
    o_x = rows.add(S, ICE, b, T, p)
    o_A = rows.add(opp(S), NON, b, p)
    return finish("a_prime", seed, rows, {"S": S, "W": W, "k": k, "L": L, "b": b, "p": p, "n": n, "T": T,
                                          "worst_oids": worst, "B_oid": o_B, "x_oid": o_x, "A_oid": o_A,
                                          "B_submission": DEPTH - 1, "x_submission": DEPTH,
                                          "A_submission": DEPTH + 1})


def _composition(rng, total, K, lo, hi):
    while True:
        parts = [int(x) for x in rng.integers(lo, hi + 1, size=K - 1)]
        last = total - sum(parts)
        if lo <= last <= hi:
            return parts + [last]


def build_b(seed):
    rng = rng_for(seed)
    S = side_of(seed)
    K = int(rng.integers(2, 5))
    ns = _composition(rng, RING, K, 2, SLICE_MARGIN) + [int(rng.integers(2, SLICE_MARGIN + 1))]   # + the burst
    P = [int(rng.integers(*MID))]
    for _ in range(K):
        P.append(better(S, P[-1], int(rng.integers(1, 21))))
    rows = Rows(rng, 2 * (K + 1))
    pairs = []
    for i, n in enumerate(ns):
        p = int(rng.integers(1, 16))
        T = ice_total(rng, n, p)
        o_ice = rows.add(S, ICE, P[i], T, p)
        o_agg = rows.add(opp(S), NON, P[i], T)
        pairs.append({"ice_oid": o_ice, "agg_oid": o_agg, "price": P[i], "n": n, "p": p, "T": T,
                      "ice_submission": 2 * i, "agg_submission": 2 * i + 1})
    return finish("b", seed, rows, {"S": S, "K": K, "pairs": pairs[:K], "burst": pairs[K]})


C_SLICES = {"c": (28, 39), "c_below": (21, 24)}   # per-iceberg slice range (finding F6): sums [112,156] / [84,96]


def build_c(seed, part="c"):
    # r2: four icebergs whose slices sum past the 100-pass bound (c) or stay below it (c_below), with the
    # analytic hop bound sum + 3 inside the generator's 160 hop margin in both.
    rng = rng_for(seed)
    S = side_of(seed)
    n_ice = 4
    lo, hi = C_SLICES[part]
    P = [int(rng.integers(*MID))]
    for _ in range(n_ice - 1):
        P.append(worse(S, P[-1], int(rng.integers(1, 21))))
    rows = Rows(rng, n_ice + 1)
    ices = []
    for j in range(n_ice):
        s, p = int(rng.integers(lo, hi + 1)), int(rng.integers(1, 31))
        T = ice_total(rng, s, p)
        ices.append({"oid": rows.add(S, ICE, P[j], T, p), "price": P[j], "n": s, "p": p, "T": T, "rank": j,
                     "submission": j})
    Q = sum(i["T"] for i in ices)
    o_agg = rows.add(opp(S), NON, P[-1], Q)
    return finish(part, seed, rows, {"S": S, "ices": ices, "agg_oid": o_agg, "agg_price": P[-1], "Q": Q,
                                     "agg_submission": n_ice, "slices_total": sum(i["n"] for i in ices),
                                     "n_trades_ring": int(PART_MARKET[part].nTrades),
                                     "pass_bound": PASS_BOUND})


def build_lr_non(seed):
    # S2 control: both sides at 100, each last row occupied by an order that neither trades nor is wiped.
    rng = rng_for(seed)
    S = side_of(seed)
    O = opp(S)
    k, L, g = int(rng.integers(1, 5)), int(rng.integers(3, 31)), int(rng.integers(1, 51))
    W = int(rng.integers(*MID))
    rows = Rows(rng, 2 * DEPTH + 1)
    worst, s_oids = _full_side(rng, rows, S, W, k, L, last_not_worst=True)
    e = better(S, W, L + g)                     # E's price: beyond S's best, so the book never crosses
    q_E = int(rng.integers(1, 51))
    o_E = rows.add(O, NON, e, q_E)
    o_others = [rows.add(O, NON, worse(O, e, int(rng.integers(1, L + 1))), int(rng.integers(1, 51)))
                for _ in range(DEPTH - 1)]     # 1..L ticks worse than e for O
    o_X = rows.add(S, NON, e, q_E)
    return finish("lr_non", seed, rows, {"S": S, "W": W, "k": k, "L": L, "e": e, "q_E": q_E,
                                         "worst_oids": worst, "E_oid": o_E, "X_oid": o_X,
                                         "last_row_S": s_oids[-1], "last_row_O": o_others[-1],
                                         "E_submission": DEPTH, "X_submission": 2 * DEPTH})


def build_lr_ice(seed):
    # S2 control: an iceberg in the first row, its side filled to 100 by NON orders (last row a NON order).
    rng = rng_for(seed)
    S = side_of(seed)
    L = int(rng.integers(3, 31))
    b = int(rng.integers(*MID))
    p, n = int(rng.integers(1, 31)), int(rng.integers(3, 11))
    T = ice_total(rng, n, p)
    rows = Rows(rng, DEPTH + 2)
    o_Z = rows.add(S, ICE, b, T, p)
    nons = [rows.add(S, NON, worse(S, b, int(rng.integers(1, L + 1))), int(rng.integers(1, 51)))
            for _ in range(DEPTH - 1)]
    v2 = min(p, T - p)
    o_A1 = rows.add(opp(S), NON, b, p)
    o_A2 = rows.add(opp(S), NON, b, v2)
    return finish("lr_ice", seed, rows, {"S": S, "L": L, "b": b, "p": p, "n": n, "T": T, "v2": v2,
                                         "Z_oid": o_Z, "A1_oid": o_A1, "A2_oid": o_A2, "last_row_S": nons[-1],
                                         "A1_submission": DEPTH, "A2_submission": DEPTH + 1})


def build_d(seed):
    rng = rng_for(seed)
    S = side_of(seed)
    M = int(PART_MARKET["d"].max_icebergs)
    P = [int(rng.integers(*MID))]
    for _ in range(M):
        P.append(worse(S, P[-1], int(rng.integers(1, 21))))
    rows = Rows(rng, M + 2)
    ices = []
    for j in range(M + 1):
        n, p = int(rng.integers(2, 9)), int(rng.integers(1, 31))
        T = ice_total(rng, n, p)
        ices.append({"oid": rows.add(S, ICE, P[j], T, p), "price": P[j], "n": n, "p": p, "T": T, "submission": j})
    o_agg = rows.add(opp(S), NON, P[0], ices[0]["T"])
    return finish("d", seed, rows, {"S": S, "M": M, "ices": ices, "overflow": ices[M], "agg_oid": o_agg,
                                    "agg_submission": M + 1})


def build_e(seed, n_slices):
    # e66 draws its parameters from e65's seed: the pair differs by exactly one slice (finding F13).
    part = "e65" if n_slices == SLICE_CEIL + 1 else "e66"
    base_seed = seed - PART_SEED_BASE[part] + PART_SEED_BASE["e65"]
    rng = rng_for(base_seed)
    S = side_of(base_seed)
    P = int(rng.integers(*MID))
    p = int(rng.integers(1, 151))
    r = int(rng.integers(1, p + 1))
    T = (n_slices - 1) * p + r
    rows = Rows(rng, 1)
    oid = rows.add(S, ICE, P, T, p)
    return finish(part, seed, rows, {"S": S, "price": P, "p": p, "r": r, "n": n_slices, "T": T, "oid": oid,
                                     "params_seed": base_seed})


BUILDERS = {"a": build_a, "a_prime": build_a_prime, "b": build_b, "c": build_c, "d": build_d,
            "e65": lambda s: build_e(s, SLICE_CEIL + 1), "e66": lambda s: build_e(s, SLICE_CEIL + 2),
            "c_below": lambda s: build_c(s, "c_below"), "lr_non": build_lr_non, "lr_ice": build_lr_ice}


def make_draw(part, seed):
    return BUILDERS[part](seed)


# ---- the dataset precondition: every structural claim of the table, re-derived from the tape alone ----------
def real_rows(tape):
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def envelope_breaches(cb):
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    return sorted(a for a in synth.AXES if int(cb[a]) > int(lim[a]))


def max_slices(tape):
    rr = real_rows(tape)
    ice = rr[rr[:, C_EXEC] == ICE]
    return max((ceil_div(r[C_QTY], r[C_PEAK]) for r in ice), default=0)


EXPECTED_ENVELOPE_BREACH = {"a": ["resting_per_side"], "a_prime": ["resting_per_side"], "b": ["trades"],
                            "c": ["trades"], "d": [], "e65": [], "e66": [], "c_below": ["trades"],
                            "lr_non": ["resting_per_side"], "lr_ice": ["resting_per_side"]}


def precondition(draw):
    bad = []
    part, prm = draw.params["part"], draw.params
    rr = real_rows(draw.tape)
    o = prm["orders"]

    def need(cond, msg):
        if not cond:
            bad.append(msg)

    need(len(rr) == len(o), f"{len(rr)} real rows for {len(o)} declared orders")
    need([[int(r[C_OID]), int(r[C_SIDE]), int(r[C_EXEC]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_PEAK]),
           int(r[C_AGENT])] for r in rr] == o, "the tape rows differ from the declared orders")
    need(len({int(r[C_OID]) for r in rr}) == len(rr), "order ids are not distinct")
    need(len({int(r[C_CONTRACT]) for r in rr}) == 1, "more than one contract id")
    for r in rr:
        oid = int(r[C_OID])
        need(0 <= int(r[C_PRICE]) <= PRICE_INT_MAX, f"oid {oid}: price outside §2.3")
        need(1 <= int(r[C_QTY]) <= QTY_INT_MAX, f"oid {oid}: quantity outside §2.3")
        need(int(r[C_PPD]) == 0 and int(r[C_EXPIRY]) == 0, f"oid {oid}: ppd or expiry not 0")
        need(int(r[C_EXEC]) in (NON, ICE), f"oid {oid}: execution restriction not NON or ICE")
        if int(r[C_EXEC]) == ICE:
            need(1 <= int(r[C_PEAK]) <= int(r[C_QTY]), f"oid {oid}: peak not in [1, total]")
        else:
            need(int(r[C_PEAK]) == 0, f"oid {oid}: NON row with a peak")
    ts = [int(r[C_TS]) for r in rr]
    need(all(b == a + 1 for a, b in zip(ts, ts[1:])), "tape timestamps not consecutive")
    cb = synth.tape_capacity_bounds(draw.tape)
    need(dict(draw.capacity_bounds) == cb, "draw.capacity_bounds is not tape_capacity_bounds(tape)")
    need(envelope_breaches(cb) == EXPECTED_ENVELOPE_BREACH[part],
         f"analytic envelope breaches {envelope_breaches(cb)} != {EXPECTED_ENVELOPE_BREACH[part]}")
    ms = max_slices(draw.tape)
    S = prm["S"]
    if part in ("a", "a_prime", "lr_non"):
        n_full = DEPTH - 1 if part == "a_prime" else DEPTH
        full = [r for r in o if r[1] == S and r[2] == NON][:n_full]
        W = prm["W"]
        need(len(full) == n_full, f"{len(full)} NON orders on the full side before the trigger, not {n_full}")
        worst_prices = [r[3] for r in full]
        need(min(worst_prices, key=lambda x: (x if S == BUY else -x)) == W, "W is not the worst price of the side")
        need(sorted(r[0] for r in full if r[3] == W) == sorted(prm["worst_oids"]), "worst-level oids mismatch")
        need(len(prm["worst_oids"]) == prm["k"] and 1 <= prm["k"] < n_full, "k mismatch")
        best_full = max(worst_prices, key=lambda x: (x if S == BUY else -x))
        if part == "lr_non":
            O = opp(S)
            others = [r for r in o if r[1] == O]
            need(len(others) == DEPTH and others[0][0] == prm["E_oid"] and others[0][3] == prm["e"]
                 and others[0][4] == prm["q_E"] and others[0][2] == NON, "E is not the first O order")
            need(all(is_better(O, prm["e"], r[3]) for r in others[1:]), "E is not strictly O's best")
            need(is_better(S, prm["e"], best_full), "O's best is not beyond S's best (the book would cross)")
            need(o[DEPTH - 1][0] == prm["last_row_S"] and o[DEPTH - 1][3] != W, "S's last-row order is at W")
            need(others[-1][0] == prm["last_row_O"] and others[-1][0] != prm["E_oid"], "O's last row is E")
            X = o[-1]
            need(X[:5] == [prm["X_oid"], S, NON, prm["e"], prm["q_E"]], "X row mismatch")
            need(len(o) == 2 * DEPTH + 1, "C14-LR-NON has not exactly 201 orders")
        elif part == "a":
            opp_row = o[0]
            need(opp_row[1] == opp(S) and is_better(S, opp_row[3], best_full), "the O order is not beyond S's best")
            over = o[DEPTH + 1]
            need(over[0] == prm["overflow_oid"] and over[1] == S and over[2] == NON, "overflow row mismatch")
            need(is_better(S, over[3], W), "the overflowing order is not strictly better than W")
            need(not is_better(S, over[3], opp_row[3]) and over[3] != opp_row[3], "the overflowing order crosses O")
            need(len(o) == DEPTH + 2, "C14a has not exactly 102 orders")
        else:
            B, x, A = o[DEPTH - 1], o[DEPTH], o[DEPTH + 1]
            need(B[0] == prm["B_oid"] and B[1] == opp(S) and B[2] == NON and B[3] == prm["b"] and B[4] == prm["p"],
                 "B row mismatch")
            need(is_better(S, B[3], best_full), "B is not beyond S's best (it would cross the full side)")
            need(x[0] == prm["x_oid"] and x[1] == S and x[2] == ICE and x[3] == prm["b"] and x[5] == prm["p"]
                 and x[4] == prm["T"], "x row mismatch")
            need(A[0] == prm["A_oid"] and A[1] == opp(S) and A[2] == NON and A[3] == prm["b"] and A[4] == prm["p"],
                 "A row mismatch")
            need(ceil_div(prm["T"], prm["p"]) == prm["n"] and 3 <= prm["n"] <= SLICE_MARGIN, "x slice count")
            need(sum(1 for r in o[:DEPTH] if r[1] == S) == DEPTH - 1, "S does not hold exactly 99 when x arrives")
            need(len(o) == DEPTH + 2, "C14a′ has not exactly 102 orders")
    elif part == "lr_ice":
        Z = o[0]
        need(Z[:6] == [prm["Z_oid"], S, ICE, prm["b"], prm["T"], prm["p"]], "Z row mismatch")
        need(ceil_div(prm["T"], prm["p"]) == prm["n"] and 3 <= prm["n"] <= SLICE_MARGIN, "Z slice count")
        nons = o[1:DEPTH]
        need(len(nons) == DEPTH - 1 and all(r[1] == S and r[2] == NON and is_better(S, prm["b"], r[3])
                                             for r in nons), "the 99 NON orders are not on S strictly behind b")
        need(nons[-1][0] == prm["last_row_S"], "last-row order mismatch")
        need(o[DEPTH][:5] == [prm["A1_oid"], opp(S), NON, prm["b"], prm["p"]], "A1 row mismatch")
        need(o[DEPTH + 1][:5] == [prm["A2_oid"], opp(S), NON, prm["b"], prm["v2"]], "A2 row mismatch")
        need(prm["v2"] == slices_of(prm["T"], prm["p"])[1], "A2 is not Z's second slice")
        need(len(o) == DEPTH + 2, "C14-LR-ICE has not exactly 102 orders")
    elif part == "b":
        pairs = prm["pairs"] + [prm["burst"]]
        need(sum(q["n"] for q in prm["pairs"]) == RING, f"pre-fill slices {sum(q['n'] for q in prm['pairs'])} != {RING}")
        for i, q in enumerate(pairs):
            need(ceil_div(q["T"], q["p"]) == q["n"] and 2 <= q["n"] <= SLICE_MARGIN, f"pair {i}: slice count")
            need(o[2 * i][:5] == [q["ice_oid"], S, ICE, q["price"], q["T"]] and o[2 * i][5] == q["p"],
                 f"pair {i}: iceberg row")
            need(o[2 * i + 1][:5] == [q["agg_oid"], opp(S), NON, q["price"], q["T"]], f"pair {i}: aggressor row")
            if i:
                need(is_better(S, q["price"], pairs[i - 1]["price"]), f"pair {i}: price not better than pair {i-1}")
        need(len(o) == 2 * len(pairs), "C14b row count")
    elif part in ("c", "c_below"):
        ices = prm["ices"]
        lo, hi = C_SLICES[part]
        for j, q in enumerate(ices):
            need(o[j][:6] == [q["oid"], S, ICE, q["price"], q["T"], q["p"]], f"iceberg {j} row")
            need(ceil_div(q["T"], q["p"]) == q["n"] and lo <= q["n"] <= hi, f"iceberg {j}: slices")
            if j:
                need(is_better(S, ices[j - 1]["price"], q["price"]), f"iceberg {j}: price not worse than {j-1}")
        need(o[-1][:5] == [prm["agg_oid"], opp(S), NON, ices[-1]["price"], prm["Q"]], "aggressor row")
        need(prm["Q"] == sum(q["T"] for q in ices), "Q != sum of totals")
        need(prm["slices_total"] + 1 < int(PART_MARKET[part].nTrades),
             f"ring {PART_MARKET[part].nTrades} does not exceed the most trades {prm['slices_total'] + 1}")
        need(int(cb["hops"]) == prm["slices_total"] + 1 + 2, "hop bound != sum of slices + 3")
        need(int(cb["hops"]) <= int(ENVELOPE.max_hops), f"hop bound {cb['hops']} outside the {ENVELOPE.max_hops} margin")
        if part == "c":
            # F6: at least sum - (n_ice - 1) passes are needed; that must exceed the 100-pass bound
            # and the loop (<= one extra slice per iceberg boundary) plus the post-loop re-submit (<= one fill
            # per iceberg) cannot consume every slice, so 0 < filled < Q
            need(prm["slices_total"] > PASS_BOUND + 2 * len(ices) - 1,
                 f"sum of slices {prm['slices_total']} not > {PASS_BOUND} + {2 * len(ices) - 1}")
        else:
            need(int(cb["hops"]) < PASS_BOUND, f"hop bound {cb['hops']} not < {PASS_BOUND}")
    elif part == "d":
        ices, M = prm["ices"], prm["M"]
        need(M == int(C14D_MAX_ICEBERGS) and len(ices) == M + 1, "C14d iceberg count != M + 1")
        for j, q in enumerate(ices):
            need(o[j][:6] == [q["oid"], S, ICE, q["price"], q["T"], q["p"]], f"iceberg {j} row")
            need(ceil_div(q["T"], q["p"]) == q["n"] and 2 <= q["n"] <= SLICE_MARGIN, f"iceberg {j}: slices")
            if j:
                need(is_better(S, ices[j - 1]["price"], q["price"]), f"iceberg {j}: price not worse than {j-1}")
        need(o[-1][:5] == [prm["agg_oid"], opp(S), NON, ices[0]["price"], ices[0]["T"]], "aggressor row")
        need(int(cb["concurrent_icebergs"]) == M + 1, "C14d: ICE rows != M + 1")
    else:   # e65 / e66
        need(len(o) == 1 and o[0][:6] == [prm["oid"], S, ICE, prm["price"], prm["T"], prm["p"]], "C14e row")
        need(ceil_div(prm["T"], prm["p"]) == prm["n"], "C14e slice count")
        need(ceil_div(prm["T"] - prm["p"], prm["p"]) == prm["n"] - 1, "C14e hidden slice count")
    if part in ("e65", "e66"):
        need(ms == prm["n"] and ms > SLICE_MARGIN, f"largest slice count {ms} not the part's {prm['n']} > {SLICE_MARGIN}")
    else:
        need(ms <= SLICE_MARGIN, f"an iceberg has {ms} > {SLICE_MARGIN} slices")
    return bad


# ---- build every draw -------------------------------------------------------------------------------------------
DRAWN = {(p, s): make_draw(p, s) for p in PARTS for s in SEEDS[p]}
PRE_BAD = {f"{LABEL[p]} seed {s}": precondition(d) for (p, s), d in DRAWN.items()}
PRE_BAD = {k: v for k, v in PRE_BAD.items() if v}


def construction_digest(part):
    h = hashlib.blake2b(digest_size=16)
    for s in SEEDS[part]:
        h.update(DRAWN[(part, s)].digest().encode("ascii"))
    return h.hexdigest()


def level_table(draw):
    # C14a / C14a′ / C14-LR-NON / C14-LR-ICE: each side's NON orders by price level (count, total qty), the
    # orders with a role in full.
    prm = draw.params
    S = prm["S"]
    W = prm.get("W")
    lv = Counter()
    qty = Counter()
    for r in prm["orders"]:
        if r[2] == NON and r[0] not in (prm.get("X_oid"), prm.get("A_oid"), prm.get("A1_oid"), prm.get("A2_oid"),
                                        prm.get("B_oid"), prm.get("opp_oid"), prm.get("overflow_oid")):
            lv[(r[1], r[3])] += 1
            qty[(r[1], r[3])] += r[4]
    head = f"**{LABEL[prm['part']]}, seed {draw.seed}** — full side {side_name(S)}"
    if W is not None:
        head += f", worst price {eur(W):.2f} EUR/MWh (int {W}), k = {prm['k']} order(s) there"
    lines = [head, "", "| side | price (EUR/MWh) | price_int | orders | qty (MW) | qty_int | worst level of the full side |",
             "|---|---|---|---|---|---|---|"]
    for sd, pr in sorted(lv, key=lambda kv: (kv[0], kv[1] if kv[0] == SELL else -kv[1])):
        lines.append(f"| {side_name(sd)} | {eur(pr):.2f} | {pr} | {lv[(sd, pr)]} | {mw(qty[(sd, pr)]):.1f} | "
                     f"{qty[(sd, pr)]} | {'yes' if (sd == S and pr == W) else ''} |")
    lines += ["", "| submission | oid | side | exec | price (EUR/MWh) | price_int | qty (MW) | qty_int | peak_int | role |",
              "|---|---|---|---|---|---|---|---|---|---|"]
    part = prm["part"]
    if part == "a":
        roles = {prm["opp_oid"]: "the O order", prm["overflow_oid"]: "the overflowing order (101st)"}
    elif part == "a_prime":
        roles = {prm["B_oid"]: "B (the only O order)", prm["x_oid"]: f"x: incoming ICE, {prm['n']} slices",
                 prm["A_oid"]: "A: consumes x's regenerated slice"}
    elif part == "lr_non":
        roles = {prm["E_oid"]: "E: O's best (the only maker)", prm["last_row_S"]: "S's last row (100th S add)",
                 prm["last_row_O"]: "O's last row (100th O add)", prm["X_oid"]: "X: the aggressor on the full own side"}
    else:
        roles = {prm["Z_oid"]: f"Z: ICE, {prm['n']} slices (first row)", prm["last_row_S"]: "S's last row (100th S add)",
                 prm["A1_oid"]: "A1: consumes Z's peak", prm["A2_oid"]: "A2: consumes Z's second slice"}
    for s, r in enumerate(prm["orders"]):
        if r[0] in roles:
            lines.append(f"| {s} | {r[0]} | {side_name(r[1])} | {'ICE' if r[2] == ICE else 'NON'} | {eur(r[3]):.2f} | "
                         f"{r[3]} | {mw(r[4]):.1f} | {r[4]} | {r[5]} | {roles[r[0]]} |")
    return "\n".join(lines)


def row_table(draw):
    prm = draw.params
    lines = [f"**{LABEL[prm['part']]}, seed {draw.seed}** — iceberg side {side_name(prm['S'])}", "",
             "| submission | oid | side | exec | price (EUR/MWh) | price_int | qty (MW) | qty_int | peak_int | slices | agent |",
             "|---|---|---|---|---|---|---|---|---|---|---|"]
    for s, r in enumerate(prm["orders"]):
        sl = ceil_div(r[4], r[5]) if r[2] == ICE else ""
        lines.append(f"| {s} | {r[0]} | {side_name(r[1])} | {'ICE' if r[2] == ICE else 'NON'} | {eur(r[3]):.2f} | "
                     f"{r[3]} | {mw(r[4]):.1f} | {r[4]} | {r[5]} | {sl} | {r[6]} |")
    return "\n".join(lines)


for _p in PARTS:
    _d = DRAWN[(_p, SEEDS[_p][0])]
    display(Markdown(level_table(_d) if _p in ("a", "a_prime", "lr_non", "lr_ice") else row_table(_d)))

print("every draw's parameters (integers; orders listed above for the first draw of each part):")
for (_p, _s), _d in DRAWN.items():
    _prm = {k: v for k, v in _d.params.items() if k not in ("orders", "market")}
    print(f"  {LABEL[_p]:8s} seed {_s}: bounds {dict(_d.capacity_bounds)}, largest slices {max_slices(_d.tape)}, "
          f"params {canon(_prm)}")
print("construction digests (blake2b-128 over each part's draw digests, seeds in order):")
for _p in PARTS:
    print(f"  {LABEL[_p]:8s} {construction_digest(_p)}")
print(f"dataset precondition: {len(DRAWN) - len(PRE_BAD)} of {len(DRAWN)} draws as described"
      + ("" if not PRE_BAD else f"; departures: {canon(PRE_BAD)}"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(16, 10.5))
for ax, _p in zip(axes.ravel(), PARTS):
    _d = DRAWN[(_p, SEEDS[_p][0])]
    _prm = _d.params
    _lv = Counter()
    _hid = Counter()
    _col = {}
    for r in _prm["orders"]:
        key = (r[1], r[3])
        vis = r[5] if r[2] == ICE else r[4]
        _lv[key] += vis
        _hid[key] += (r[4] - r[5]) if r[2] == ICE else 0
    for (side, pr), v in _lv.items():
        sgn = 1 if side == BUY else -1
        c = "tab:green" if side == BUY else "tab:red"
        if pr == _prm.get("W") and side == _prm["S"]:
            c = "black"
        ax.barh(eur(pr), mw(v) * sgn, height=0.006 * max(1, len(_lv)) ** 0.5, color=c)
        if _hid[(side, pr)]:
            ax.barh(eur(pr), mw(_hid[(side, pr)]) * sgn, left=mw(v) * sgn, height=0.006 * max(1, len(_lv)) ** 0.5,
                    color="none", edgecolor=c, hatch="//")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{LABEL[_p]} (seed {_d.seed}): {CEILING_OF[_p]}", fontsize=9)
    ax.set_xlabel("MW (buy right, sell left; hatched = hidden)", fontsize=8)
    ax.set_ylabel("limit price (EUR/MWh)", fontsize=8)
    ax.tick_params(labelsize=7)
for _ax in axes.ravel()[len(PARTS):]:
    _ax.axis("off")
axes.ravel()[-1].text(0.0, 0.5, "All rows of each tape, by price level, in the order-book frame\n"
                      "(aggressors drawn at their limit). Black: the full side's worst level W.\n"
                      "Hatched: an iceberg's hidden remainder (total - peak).", fontsize=9, va="center")
plt.tight_layout()
plt.show()

## 5. Known result

`expectation(draw)` below computes, from `draw.params` alone (never from the generator's own expectation, never from a run), everything section 7 compares. Notation: $S$ the full (or iceberg) side, $O$ the other side; submission $s$ is the $s$-th real row; "post $s$" is the book right after submission $s$ returns; an iceberg of total $T$ and peak $p$ has slices $\mathrm{sl}(T,p) = (p, \ldots, p, T-(n-1)p)$, $n = \lceil T/p \rceil$ (§4.2); a trade is written $[\text{submission}, \text{ring row}, \text{price}, \text{qty}, \text{buyer oid}, \text{seller oid}]$ and prints at the resting order's (or the iceberg's) price (§6.2 items 3 and 7). Every order is accepted (§2.3, checked in section 4), nothing is cancelled (no IOC, FOK, cancellation or gate expiry is drawn), and the reported quantity of a submission is the engine's report in ticks: the **true** total it traded (finding F4, corrected), which equals its rows' total everywhere except on C14b's burst.

**C14a — book depth (certified).** Posts 0–100 hold $0, 1, \ldots, 100$ orders on $S$; `book_full` is false on every post up to 100 (finding F10). Submission 101 (the overflow, priced better than $W$) raises `book_full` and wipes **every** order at $W$ (OQ-10, review §5 item 3):
$$\mathrm{evicted} = \{o : \text{price}(o) = W\} \text{ at full quantity},\quad |S| \text{ after post }101 = 100 - k + 1 .$$
No trade, no hidden quantity, no `truncated`. Every other order rests untouched. I8 breach set $=\{\texttt{resting\_per\_side}\,(=100), \texttt{book\_full}\}$.

**C14a′ — full only at regeneration (recorded, not certified; finding F16).** Posts 0–98: $1, \ldots, 99$ on $S$; post 99 (B on $O$): still 99. Submission 100: x's peak trades $p$ against B at $b$ (B rests, so B's price; §6.2 item 3): one trade $[100, 0, b, p, \cdot, \cdot]$; B is filled; x's regenerated slice $p$ (since $n \ge 3$) rests at $b$ and brings $S$ to **100**; every dispatch of x met a side at 99, so no flag site fired (correction 1a). Submission 101: A consumes that slice, one trade $[101, 1, b, p, \cdot, \cdot]$; x's regeneration is then the only dispatch on $S$; its consumed slice has left the book, so it meets 99 other orders and rests $v_3 = \min(p, T - 2p)$ without a wipe. So
$$\mathrm{evicted} = \varnothing,\quad \text{filled}(x) = 2p,\quad \text{resting}(x) = v_3,\quad \text{hidden}(x) = T - 2p - v_3,\quad |S| = 100 \text{ on posts } 100, 101,$$
and `book_full` is false on **every** post. I8 breach set $=\{\texttt{resting\_per\_side}\,(=100)\}$: no flag, the resting axis still sees the side at 100. Any eviction or other lost quantity is a recorded departure.

**C14b — the trade ring.** The pre-fill writes $\sum_i n_i = 100$ trades: pair $i$'s aggressor (submission $2i+1$) writes $\mathrm{sl}(T_i, p_i)$ at $P_i$, in ring rows $\sum_{i'<i} n_{i'}, \ldots$; the ring is then exactly full. The burst aggressor's $n_b$ trades all land in row 99 (OQ-15), so the ring keeps **only the final one**: row 99 $= [P_b, r_b, \ldots]$, $r_b = T_b - (n_b - 1)p_b$; rows 0–98 are the first 99 pre-fill trades; the 100th pre-fill trade and the burst's first $n_b - 1$ trades are gone from the ring. The record therefore holds, for the burst submission, one fill $[2K+1, 99, P_b, r_b, \ldots]$, while the engine reports the burst's **true** quantity $T_b$ (finding F4, corrected). Rows used per post: $0, n_1, n_1, n_1+n_2, \ldots, 100, 100, 100$. Every order fully traded (nothing rests, no hidden). Flags: `trade_buffer_saturated` true; `book_full` and `truncated` false. I8 breach set $=\{\texttt{trades}\,(=100), \texttt{trade\_buffer\_saturated}\}$. **Excluded:** I2 fails on exactly ICE$_b$ and its aggressor, each by $\mathrm{lhs} = r_b \ne T_b$ (`identity`); I6 fails on exactly submission $2K+1$: its quantity violation (expected $r_b$ from the rows, observed $T_b$ reported) is required, a settlement-value violation on the same submission is admitted, nothing else; I1 **passes**.

**C14c — the 100-pass bound.** Post 3 (the last registration): `truncated` false (each $s_j \le 39$). The aggressor needs at least $\sum_j s_j - 3 > 100$ passes (finding F6), so post 4 raises `truncated` and the walk stops short: $0 < \mathrm{filled}(\text{aggressor}) < Q$. Every fill is $[4, \cdot, P_j, q, \text{buyer}, \text{seller}]$ against ICE$_j$ with $q \le p_j$; the fills are the documented walk $E = \mathrm{sl}(T_1,p_1)@P_1, \mathrm{sl}(T_2,p_2)@P_2, \ldots$ up to an unobservable cut point, followed by at most one trade per later iceberg against its visible slice, in price order (the post-loop re-submit; ladder out-of-scope note). **Full outcome** (recorded whole; asserted consistent with the observed fills $F$): `filled` is the per-order sum of $F$; `reported_qty` $= [0,0,0,0,\mathrm{filled}(\text{aggressor})]$; the aggressor rests $Q - \mathrm{filled}(\text{aggressor})$ at $P_4$ (§4.5 NON) with no hidden; each ICE$_j$ rests at most $p_j$ at $P_j$, and `hidden_remaining` names icebergs only; `final_book` is exactly `still_resting` at each order's declared side and price. I2 closes for every order, nothing evicted or cancelled, no `book_full`, ring not saturated. **Excluded:** I3 — the final book shows the Rule-4 departure (visibly crossed, **or** a consumed slot: an iceberg with hidden quantity and no visible slice opposite the aggressor's resting remainder); I3 fails exactly when it is visibly crossed, and only on post 4 or the final book (finding F7). I8 breach set $\supseteq \{\texttt{truncated}\}$, $\subseteq$ that $\cup \{\texttt{trades}\}$ (the trade count before the cut is unobservable), with the analytic hop bound $\sum s_j + 3$ pinned and **not** a breach (it lies inside the 160 margin).

**C14c-below — control under the bound.** Same construction, $\sum_j s_j \in [84, 96]$, sound hop bound $\sum_j s_j + 3 \le 99 < 100$: `truncated` false on every post; the aggressor completes the whole walk $E$, one trade per slice, in ring rows $0, \ldots, \sum s_j - 1$, all on submission 4; every order filled exactly ($T_j$, and $Q$); the book ends empty; `reported_qty` $= [0,0,0,0,Q]$. I8 breach set $=\{\texttt{trades}\,(=\sum s_j)\}$, hop bound pinned. Every battery check passes, I3 included.

**C14d — iceberg slots (cap $M$).** Posts 0…$M-1$: $1, \ldots, M$ live icebergs, `truncated` false. Post $M$ (the $(M{+}1)$-th iceberg): `truncated` true, live icebergs still $M$, the side channel byte-identical to its state before the submission (§B1); its peak $p_{M+1}$ rests at $P_{M+1}$, its hidden remainder is tracked nowhere (finding F9). Submission $M{+}1$ sweeps ICE$_1$: $\mathrm{sl}(T_1, p_1)$ at $P_1$ in rows $0, 1, \ldots$; `truncated` stays true. Final: ICE$_2$…ICE$_M$ rest $p_j$ with hidden $T_j - p_j$; ICE$_{M+1}$ rests $p_{M+1}$, hidden 0. I8 breach set $=\{\texttt{truncated}\}$. **Excluded:** I2 fails on exactly ICE$_{M+1}$, by $\mathrm{lhs} = p_{M+1} \ne T_{M+1}$ (`identity`).

**C14e — slices per iceberg.** $\lceil (T-p)/p \rceil = n - 1$: 64 for the 65-slice iceberg (not $> 64$: `truncated` false), 65 for the 66-slice one (`truncated` true on post 0, at registration). In both: no trade; the book holds exactly $[P, p, \text{oid}]$ on $S$; hidden $= T - p$; I2 closes. I8 breach set $=\{\texttt{slices}\,(=n)\}$ for 65, $\{\texttt{slices}, \texttt{truncated}\}$ for 66 (finding F12).

**C14-LR-NON — crossing NON, both last rows occupied (control, S2).** Posts 0–99: $S$ holds $1, \ldots, 100$, $O$ none; posts 100–199: $S$ 100, $O$ $1, \ldots, 100$; `book_full` false throughout (each 100th add met 99). Submission 200: X trades $q_E$ against E at $e$, one trade $[200, 0, e, q_E, \cdot, \cdot]$; X executes completely, but its own side $S$ was full, so the wipe fires on that dispatch: `book_full` raised on post 200, every order at $W$ evicted at full quantity (correction 1a). Nothing else is lost: both last-row occupants and every other order rest untouched; $|S| = 100 - k$, $|O| = 99$ after post 200. I8 breach set $=\{\texttt{resting\_per\_side}\,(=100), \texttt{book\_full}\}$.

**C14-LR-ICE — iceberg regeneration with the last row occupied (control, S2).** Post 0: $S$ holds Z's peak; posts 1–99: $2, \ldots, 100$; `book_full` false throughout. Submission 100: A1 consumes Z's peak, one trade $[100, 0, b, p, \cdot, \cdot]$; Z regenerates its second slice $v_2 = p$ at $b$ (a miss is a miss: Z is not stranded). Submission 101: A2 reaches that slice, one trade $[101, 1, b, v_2, \cdot, \cdot]$; Z regenerates $v_3 = \min(p, T - 2p)$, hidden $T - 2p - v_3$. $|S| = 100$ on posts 100 and 101; nothing evicted, no flag. I8 breach set $=\{\texttt{resting\_per\_side}\,(=100)\}$.

**The battery** (section 7 runs RB and I1–I9 on the jitted record; I2 with `allow_eviction=True` on C14a, C14a′ and C14-LR-NON only; I8 with `allow_breach=True`). The statuses every **non-excluded** check must reach (`n/a` = `not_applicable`; under the judging rule a `not_applicable` result also satisfies an expected `pass`):

| part | RB | I1 | I2 | I3 | I4 | I5 | I6 | I7 | I9 (`expects_trade`) |
|---|---|---|---|---|---|---|---|---|---|
| C14a | pass | n/a | pass | pass | n/a | n/a | pass | pass | pass (no) |
| C14a′ | pass | pass | pass | pass | pass | pass | pass | pass | pass (yes) |
| C14b | pass | *excl.* | *excl.* | pass | pass | pass | *excl.* | pass | pass (yes) |
| C14c | pass | pass | pass | *excl.* | pass | pass | pass | pass | pass (yes) |
| C14c-below | pass | pass | pass | pass | pass | pass | pass | pass | pass (yes) |
| C14d | pass | pass | *excl.* | pass | pass | pass | pass | pass | pass (yes) |
| C14e | pass | n/a | pass | pass | n/a | n/a | pass | pass | pass (no) |
| C14-LR-NON | pass | pass | pass | pass | pass | pass | pass | pass | pass (yes) |
| C14-LR-ICE | pass | pass | pass | pass | pass | pass | pass | pass | pass (yes) |

I8 is excluded on every part; its complete breach set (the battery's axes and flags, plus the slice margin re-checked from the tape) is asserted exactly as stated above.

In [ ]:
NA = "not_applicable"
BATTERY_IDS = ("RB", "I1", "I2", "I3", "I4", "I5", "I6", "I7", "I8", "I9")


def nz(m):
    # A per-order map, zero entries dropped, keys as strings, sorted by order id.
    return {str(int(k)): int(v) for k, v in sorted(m.items(), key=lambda kv: int(kv[0])) if int(v)}


def book_json(entries):
    # entries: (side, price_int, qty_int, oid) of every resting order -> {"BUY": [...], "SELL": [...]}, sorted.
    out = {"BUY": [], "SELL": []}
    for side, p, q, o in entries:
        out[side_name(side)].append([int(p), int(q), int(o)])
    return {k: sorted(v) for k, v in out.items()}


def buyer_seller(a_oid, a_side, b_oid):
    # The buyer and the seller of a trade between a_oid (on a_side) and b_oid (the other side).
    return (a_oid, b_oid) if int(a_side) == BUY else (b_oid, a_oid)


def battery_row(no_fill):
    return {"RB": "pass", "I1": NA if no_fill else "pass", "I2": "pass", "I3": "pass",
            "I4": NA if no_fill else "pass", "I5": NA if no_fill else "pass", "I6": "pass", "I7": "pass",
            "I9": "pass"}


def exp_a(prm):
    o, S, worst = prm["orders"], prm["S"], set(prm["worst_oids"])
    n = len(o)
    surv = [r for r in o if r[0] not in worst]
    rows_S = [sum(1 for r in o[1:s + 1] if r[1] == S) for s in range(n - 1)] + [DEPTH - len(worst) + 1]
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [], "fills": [], "filled": {},
                    "still_resting": nz({r[0]: r[4] for r in surv}), "hidden_remaining": {},
                    "evicted": nz({r[0]: r[4] for r in o if r[0] in worst}), "cancelled": {},
                    "final_book": book_json([(r[1], r[3], r[4], r[0]) for r in surv]), "reported_qty": [0] * n},
        "flags": {"book_full_posts": [n - 1], "truncated_posts": [], "trade_buffer_saturated": False},
        "extra": {"full_side_rows_posts": rows_S},
        "i8": {"required": ["book_full", "resting_per_side"], "allowed": ["book_full", "resting_per_side"],
               "pinned": {"resting_per_side": DEPTH}},
        "battery": battery_row(no_fill=True),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": False, "allow_eviction": True,
    }


def exp_a_prime(prm):
    # r2 (finding F16): every flag site meets a side at 99; the regenerations replace the row their own
    # consumed slice vacated, so nothing is evicted and no flag is raised.
    o, S = prm["orders"], prm["S"]
    n = len(o)
    p, T, b = prm["p"], prm["T"], prm["b"]
    B, x, A = prm["B_oid"], prm["x_oid"], prm["A_oid"]
    v3 = min(p, T - 2 * p)
    hidden = T - 2 * p - v3
    nons = [r for r in o[:DEPTH - 1]]
    xB_buyer, xB_seller = buyer_seller(x, S, B)
    xA_buyer, xA_seller = buyer_seller(x, S, A)
    rows_S = [s + 1 for s in range(DEPTH - 1)] + [DEPTH - 1, DEPTH, DEPTH]
    rep = [0] * n
    rep[prm["x_submission"]] = p
    rep[prm["A_submission"]] = p
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [],
                    "fills": [[prm["x_submission"], 0, b, p, xB_buyer, xB_seller],
                              [prm["A_submission"], 1, b, p, xA_buyer, xA_seller]],
                    "filled": nz({B: p, A: p, x: 2 * p}),
                    "still_resting": nz({**{r[0]: r[4] for r in nons}, x: v3}),
                    "hidden_remaining": nz({x: hidden}),
                    "evicted": {}, "cancelled": {},
                    "final_book": book_json([(r[1], r[3], r[4], r[0]) for r in nons] + [(S, b, v3, x)]),
                    "reported_qty": rep},
        "flags": {"book_full_posts": [], "truncated_posts": [], "trade_buffer_saturated": False},
        "extra": {"full_side_rows_posts": rows_S},
        "i8": {"required": ["resting_per_side"], "allowed": ["resting_per_side"],
               "pinned": {"resting_per_side": DEPTH}},
        "battery": battery_row(no_fill=False),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": True,
    }


def exp_b(prm):
    o, S = prm["orders"], prm["S"]
    n = len(o)
    pairs, burst = prm["pairs"], prm["burst"]
    fills, ring, used, rep, filled = [], [], [], [0] * n, {}
    row = 0
    for q in pairs:
        buyer, seller = buyer_seller(q["ice_oid"], S, q["agg_oid"])
        used.append(row)                                   # post of the iceberg's submission
        for sl in slices_of(q["T"], q["p"]):
            fills.append([q["agg_submission"], row, q["price"], sl, buyer, seller])
            ring.append([q["price"], sl, buyer, seller])
            row += 1
        used.append(row)                                   # post of the aggressor's submission
        rep[q["agg_submission"]] = q["T"]
        filled[q["ice_oid"]] = q["T"]
        filled[q["agg_oid"]] = q["T"]
    b_buyer, b_seller = buyer_seller(burst["ice_oid"], S, burst["agg_oid"])
    r_b = slices_of(burst["T"], burst["p"])[-1]
    fills.append([burst["agg_submission"], RING - 1, burst["price"], r_b, b_buyer, b_seller])
    ring = ring[:RING - 1] + [[burst["price"], r_b, b_buyer, b_seller]]
    used += [RING, RING]
    rep[burst["agg_submission"]] = burst["T"]          # the engine reports the TRUE burst quantity (F4, r2)
    filled[burst["ice_oid"]] = r_b
    filled[burst["agg_oid"]] = r_b
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [], "fills": fills, "filled": nz(filled),
                    "still_resting": {}, "hidden_remaining": {}, "evicted": {}, "cancelled": {},
                    "final_book": book_json([]), "reported_qty": rep},
        "flags": {"book_full_posts": [], "truncated_posts": [], "trade_buffer_saturated": True},
        "extra": {"ring": ring, "rows_used_posts": used},
        "i8": {"required": ["trade_buffer_saturated", "trades"], "allowed": ["trade_buffer_saturated", "trades"],
               "pinned": {"trades": RING}},
        "battery": {k: "pass" for k in ("RB", "I3", "I4", "I5", "I7", "I9")},
        "excluded": {"I1": {"kind": "status", "status": "pass"},
                     "I2": {"kind": "status", "status": "fail",
                            "violations": sorted([[burst["ice_oid"], "identity", burst["T"], r_b],
                                                  [burst["agg_oid"], "identity", burst["T"], r_b]])},
                     # I6(b): exactly the burst submission departs; its quantity violation (rows r_b, reported
                     # T_b) is required, a settlement-value violation on the same submission is admitted (F4)
                     "I6": {"kind": "i6_burst", "status": "fail", "submission": burst["agg_submission"],
                            "qty_violation": [burst["agg_submission"], "engine_reports", r_b, burst["T"]]},
                     "I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": False,
    }


def exp_c(prm):
    # r2: the 100-pass bound. The cut point is unobservable, so the fills are checked against the walk (section 7)
    # and the full outcome is recorded whole and checked for consistency with the observed fills ("closure").
    o, S, ices = prm["orders"], prm["S"], prm["ices"]
    agg = prm["agg_oid"]
    seq = []
    for q in ices:
        buyer, seller = buyer_seller(q["oid"], S, agg)
        for sl in slices_of(q["T"], q["p"]):
            seq.append([q["price"], sl, buyer, seller])
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [], "evicted": {}, "cancelled": {}},
        "flags": {"book_full_posts": [], "truncated_posts": [prm["agg_submission"]], "trade_buffer_saturated": False},
        "extra": {},
        "i8": {"required": ["truncated"], "allowed": ["trades", "truncated"],
               "pinned": {"hops": prm["slices_total"] + 1 + 2}},
        "battery": {k: "pass" for k in ("RB", "I1", "I2", "I4", "I5", "I6", "I7", "I9")},
        "excluded": {"I3": {"kind": "c_i3"}, "I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": False,
        "walk": {"sequence": seq, "agg": agg, "agg_side": opp(S), "Q": prm["Q"], "agg_submission": prm["agg_submission"],
                 "agg_price": prm["agg_price"], "n_orders": len(o),
                 "ices": {str(q["oid"]): {"price": q["price"], "p": q["p"], "T": q["T"], "rank": q["rank"]} for q in ices},
                 "S": S},
    }


def exp_c_below(prm):
    # Control: the sound hop bound is < 100, so the walk completes exactly and nothing is truncated.
    o, S, ices = prm["orders"], prm["S"], prm["ices"]
    agg, s_agg, Q = prm["agg_oid"], prm["agg_submission"], prm["Q"]
    fills, row = [], 0
    for q in ices:
        buyer, seller = buyer_seller(q["oid"], S, agg)
        for sl in slices_of(q["T"], q["p"]):
            fills.append([s_agg, row, q["price"], sl, buyer, seller])
            row += 1
    rep = [0] * len(o)
    rep[s_agg] = Q
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [], "fills": fills,
                    "filled": nz({**{q["oid"]: q["T"] for q in ices}, agg: Q}), "still_resting": {},
                    "hidden_remaining": {}, "evicted": {}, "cancelled": {}, "final_book": book_json([]),
                    "reported_qty": rep},
        "flags": {"book_full_posts": [], "truncated_posts": [], "trade_buffer_saturated": False},
        "extra": {},
        "i8": {"required": ["trades"], "allowed": ["trades"],
               "pinned": {"hops": prm["slices_total"] + 1 + 2, "trades": prm["slices_total"]}},
        "battery": battery_row(no_fill=False),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": False,
    }


def exp_lr_non(prm):
    # S2 control: a crossing NON on a full own side; the documented wipe (correction 1a) and nothing else.
    o, S, worst = prm["orders"], prm["S"], set(prm["worst_oids"])
    O = opp(S)
    n = len(o)
    E, X, e, q_E = prm["E_oid"], prm["X_oid"], prm["e"], prm["q_E"]
    s_X = prm["X_submission"]
    buyer, seller = buyer_seller(X, S, E)
    surv = [r for r in o[:s_X] if r[0] not in worst and r[0] != E]
    rep = [0] * n
    rep[s_X] = q_E
    rows_S = [s + 1 for s in range(DEPTH)] + [DEPTH] * DEPTH + [DEPTH - len(worst)]
    rows_O = [0] * DEPTH + [s + 1 for s in range(DEPTH)] + [DEPTH - 1]
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [],
                    "fills": [[s_X, 0, e, q_E, buyer, seller]],
                    "filled": nz({E: q_E, X: q_E}),
                    "still_resting": nz({r[0]: r[4] for r in surv}),
                    "hidden_remaining": {},
                    "evicted": nz({r[0]: r[4] for r in o if r[0] in worst}), "cancelled": {},
                    "final_book": book_json([(r[1], r[3], r[4], r[0]) for r in surv]),
                    "reported_qty": rep},
        "flags": {"book_full_posts": [s_X], "truncated_posts": [], "trade_buffer_saturated": False},
        "extra": {"full_side_rows_posts": rows_S, "other_side_rows_posts": rows_O,
                  "last_rows_resting": {"S": [prm["last_row_S"], True], "O": [prm["last_row_O"], True]}},
        "i8": {"required": ["book_full", "resting_per_side"], "allowed": ["book_full", "resting_per_side"],
               "pinned": {"resting_per_side": DEPTH}},
        "battery": battery_row(no_fill=False),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": True,
    }


def exp_lr_ice(prm):
    # S2 control: the iceberg regenerates after each consumption although its side's last row is occupied.
    o, S = prm["orders"], prm["S"]
    n = len(o)
    Z, A1, A2, p, T, b, v2 = prm["Z_oid"], prm["A1_oid"], prm["A2_oid"], prm["p"], prm["T"], prm["b"], prm["v2"]
    v3 = min(p, T - p - v2)
    hidden = T - p - v2 - v3
    nons = o[1:DEPTH]
    b1, s1 = buyer_seller(Z, S, A1)
    b2, s2 = buyer_seller(Z, S, A2)
    rep = [0] * n
    rep[prm["A1_submission"]] = p
    rep[prm["A2_submission"]] = v2
    rows_S = [s + 1 for s in range(DEPTH)] + [DEPTH, DEPTH]
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [],
                    "fills": [[prm["A1_submission"], 0, b, p, b1, s1], [prm["A2_submission"], 1, b, v2, b2, s2]],
                    "filled": nz({Z: p + v2, A1: p, A2: v2}),
                    "still_resting": nz({**{r[0]: r[4] for r in nons}, Z: v3}),
                    "hidden_remaining": nz({Z: hidden}),
                    "evicted": {}, "cancelled": {},
                    "final_book": book_json([(r[1], r[3], r[4], r[0]) for r in nons] + [(S, b, v3, Z)]),
                    "reported_qty": rep},
        "flags": {"book_full_posts": [], "truncated_posts": [], "trade_buffer_saturated": False},
        "extra": {"full_side_rows_posts": rows_S,
                  "last_rows_resting": {"S": [prm["last_row_S"], True]}},
        "i8": {"required": ["resting_per_side"], "allowed": ["resting_per_side"], "pinned": {"resting_per_side": DEPTH}},
        "battery": battery_row(no_fill=False),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": False,
    }


def exp_d(prm):
    o, S, ices, M = prm["orders"], prm["S"], prm["ices"], prm["M"]
    agg, first, over = prm["agg_oid"], ices[0], prm["overflow"]
    buyer, seller = buyer_seller(first["oid"], S, agg)
    fills = [[prm["agg_submission"], j, first["price"], sl, buyer, seller]
             for j, sl in enumerate(slices_of(first["T"], first["p"]))]
    rep = [0] * len(o)
    rep[prm["agg_submission"]] = first["T"]
    rest = ices[1:]
    return {
        "outcome": {"accepted": sorted(r[0] for r in o), "rejected": [], "fills": fills,
                    "filled": nz({first["oid"]: first["T"], agg: first["T"]}),
                    "still_resting": nz({q["oid"]: q["p"] for q in rest}),
                    "hidden_remaining": nz({q["oid"]: q["T"] - q["p"] for q in ices[1:M]}),
                    "evicted": {}, "cancelled": {},
                    "final_book": book_json([(S, q["price"], q["p"], q["oid"]) for q in rest]), "reported_qty": rep},
        "flags": {"book_full_posts": [], "truncated_posts": [M, M + 1], "trade_buffer_saturated": False},
        "extra": {"n_active_posts_through_overflow": list(range(1, M + 1)) + [M], "meta_unchanged_at_overflow": True},
        "i8": {"required": ["truncated"], "allowed": ["truncated"], "pinned": {}},
        "battery": {k: "pass" for k in ("RB", "I1", "I3", "I4", "I5", "I6", "I7", "I9")},
        "excluded": {"I2": {"kind": "status", "status": "fail",
                            "violations": [[over["oid"], "identity", over["T"], over["p"]]]},
                     "I8": {"kind": "i8"}},
        "expects_trade": True, "allow_eviction": False,
    }


def exp_e(prm):
    S, oid, p, T, n = prm["S"], prm["oid"], prm["p"], prm["T"], prm["n"]
    over = ceil_div(T - p, p) > SLICE_CEIL          # the precondition's formula (finding F13)
    req = ["slices", "truncated"] if over else ["slices"]
    return {
        "outcome": {"accepted": [oid], "rejected": [], "fills": [], "filled": {}, "still_resting": nz({oid: p}),
                    "hidden_remaining": nz({oid: T - p}), "evicted": {}, "cancelled": {},
                    "final_book": book_json([(S, prm["price"], p, oid)]), "reported_qty": [0]},
        "flags": {"book_full_posts": [], "truncated_posts": [0] if over else [], "trade_buffer_saturated": False},
        "extra": {},
        "i8": {"required": req, "allowed": req, "pinned": {"slices": n}},
        "battery": battery_row(no_fill=True),
        "excluded": {"I8": {"kind": "i8"}},
        "expects_trade": False, "allow_eviction": False,
    }


EXPECT = {"a": exp_a, "a_prime": exp_a_prime, "b": exp_b, "c": exp_c, "d": exp_d, "e65": exp_e, "e66": exp_e,
          "c_below": exp_c_below, "lr_non": exp_lr_non, "lr_ice": exp_lr_ice}


def expectation(draw):
    e = EXPECT[draw.params["part"]](draw.params)
    e["part"] = draw.params["part"]
    # every check id is either asserted by status (non-excluded) or by its excluded spec, never both, never neither
    assert set(e["battery"]) | set(e["excluded"]) == set(BATTERY_IDS), e["part"]
    assert not set(e["battery"]) & set(e["excluded"]), e["part"]
    return e


EXPECTED = {key: expectation(d) for key, d in DRAWN.items()}

for _p in PARTS:
    _s = SEEDS[_p][0]
    _e = EXPECTED[(_p, _s)]
    _o = _e["outcome"]
    print(f"{LABEL[_p]} seed {_s}: fills {len(_o.get('fills', [])) if 'fills' in _o else 'not pinned (cut point unobservable)'}, "
          f"evicted {_o['evicted']}, flags {canon(_e['flags'])}, I8 required {_e['i8']['required']} "
          f"pinned {_e['i8']['pinned']}, excluded {canon(_e['excluded'])}")
    if _p in ("e65", "e66"):
        _prm = DRAWN[(_p, _s)].params
        print(f"    T = {_prm['T']}, p = {_prm['p']}: ceil(T/p) = {ceil_div(_prm['T'], _prm['p'])}, "
              f"ceil((T-p)/p) = {ceil_div(_prm['T'] - _prm['p'], _prm['p'])} -> truncated "
              f"{'true' if _e['flags']['truncated_posts'] else 'false'}")
    if _p == "b":
        _prm = DRAWN[(_p, _s)].params
        print(f"    pre-fill slices {[q['n'] for q in _prm['pairs']]} (sum {sum(q['n'] for q in _prm['pairs'])}), "
              f"burst {_prm['burst']['n']} slices of {_prm['burst']['p']}: ring row 99 keeps {canon(_e['extra']['ring'][-1])}; "
              f"engine reports T_b = {_prm['burst']['T']} (rows: {_e['extra']['ring'][-1][1]})")
    if _p in ("c", "c_below"):
        _prm = DRAWN[(_p, _s)].params
        print(f"    slices {[q['n'] for q in _prm['ices']]} (sum {_prm['slices_total']}; pass bound {PASS_BOUND}, "
              f"hop bound {_prm['slices_total'] + 3}, hop margin {ENVELOPE.max_hops}); Q = {_prm['Q']} ticks")

## 6. Run

### The engine, eager and jitted, one fresh book per draw (wired at K3)

> **Wired at K3 (r2, 2026-10-07).** The code cell below defines `run_part` exactly as the contract states: the battery's `build_record` on the draw's tape with `PART_MARKET[part]` (C14c and C14c-below with a 256-row ring, C14d with `max_icebergs` 4, every other part the default configuration) and the draw's analytic bounds, nothing else. Sections 1–5 and 7 are the expectation and are not edited here (independence rule 2).

Section 7 consumes exactly **one name** from this section:

1. **`run_part(draw, *, jit) -> RunRecord`**: the battery's record builder on the draw's tape, with the draw's part configuration and analytic bounds, i.e. exactly
   `tests.omiecid.reference.omie_invariants.build_record(draw.tape, market_config=PART_MARKET[draw.params["part"]], capacity_bounds=dict(draw.capacity_bounds), jit=jit)`,
   with no other argument: one fresh book of an OPEN contract (the tape's contract id), the default `OMIEProductConfig()`, no cancellation, no gate expiry. `draw` is always a `Draw` built by section 4's `make_draw(part, seed)` (all of them are in `DRAWN`). `jit=False` runs eager (`jax.disable_jit()`), `jit=True` jitted. Every call starts from its own fresh book; the function may raise, and section 7 records a raise as a failed check.

Section 7 calls `run_part(draw, jit=True)` on every draw, and `run_part(draw, jit=False)` on the first `EAGER_PER_PART` draws of every part. It reads these `RunRecord` fields: `accepted` (keys), `rejections` (each `oid`), `fills` (each `submission`, `row_index`, `price_int`, `qty_int`, `buyer_oid`, `seller_oid`), `filled`, `still_resting`, `hidden_remaining`, `evicted`, `cancelled_or_expired`, `engine_reports` (each `trade_qty_mw`), `pre_snapshots` / `post_snapshots` / `final_snapshot` (`resting(side)`, `book_full`, `truncated`, `n_active`, `iceberg_meta`, `trades`, `trade_rows_used()`, `visible_qty`, `best_bid()`, `best_ask()`), `flags`, `observed_capacity`, `tape`; and the whole record through the battery's checks and `compare_records`.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_part(draw, *, jit) -> RunRecord, per the contract above.
#
# run_part: the battery's record builder on the draw's tape, with the draw's part market
# configuration (PART_MARKET) and the draw's analytic capacity bounds, and no other argument:
# one fresh book of an OPEN contract, the default OMIEProductConfig(), no cancellation, no
# gate expiry. jit=False runs eager (build_record wraps the whole build in jax.disable_jit()),
# jit=True jitted. Every call builds its own fresh book; a raise propagates to section 7,
# which records it as a failed check.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import build_record


def run_part(draw, *, jit):
    """The battery's record builder on `draw`'s tape, with its part configuration and analytic bounds."""
    return build_record(draw.tape, market_config=PART_MARKET[draw.params["part"]],
                        capacity_bounds=dict(draw.capacity_bounds), jit=jit)


print("run_part wired: build_record(tape, market_config=PART_MARKET[part], capacity_bounds=draw.capacity_bounds, jit)")


## 7. Certification

The oracle comparisons come first, per draw and per part (section 5); then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the draw accounting. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per draw** (expected and observed as canonical compact JSON, sorted keys, **in full**, never truncated; the `detail` names the differing fields):

1. `engine outcome`: accepted and rejected ids, every fill `[submission, ring row, price, qty, buyer, seller]` in record order, `filled`, `still_resting`, `hidden_remaining`, `evicted`, `cancelled_or_expired`, the final book on both sides, and the engine's reported quantity per submission in ticks (C14c: accepted, rejected, evicted and cancelled exactly; its fills are item 4 and the rest of its outcome item 4a).
2. `flags`: the posts on which `book_full` and `truncated` are raised, and the record's `trade_buffer_saturated`.
3. `part detail`: C14a/C14a′ the full side's row count on every post; C14b the whole ring after the run (row by row) and the rows used on every post; C14d the live-iceberg count on every post through the overflow and whether the side channel is byte-identical across the overflowing submission; C14-LR-NON/C14-LR-ICE each side's row count on every post and whether each last-row occupant still rests.
4. C14c `walk`: the observed fills against the full documented walk (prefix length, the post-cut tail and its rule), and $0 < \mathrm{filled}(\text{aggressor}) < Q$.
4a. C14c `full outcome`: `filled`, `still_resting`, `hidden_remaining`, `final_book`, `reported_qty` (and the fills) recorded whole, and checked consistent with the observed fills (section 5's closure rules).
5. `battery (non-excluded)`: each non-excluded check's status against section 5's table (a `not_applicable` result satisfies an expected `pass`), with every violation message of a check that departs.
6. `excluded <check>`: each excluded check's actual result against its stated expectation — I8's complete breach set (the battery's axes and flags plus the slice margin) and the pinned values; C14b's I2 violation set (oid, field, expected, observed), its I1 status, and its I6 violation set (the burst submission's quantity violation required, at most one settlement-value violation on that submission admitted); C14c's Rule-4 departure (visible cross or consumed slot), the I3 status and where its violations sit; C14d's I2 violation set.

The battery is folded into the invariant table with every excluded check shown as `not_applicable` (reason: asserted separately), so the table counts the non-excluded checks only; the excluded checks' assertions are items 6. The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; the five configured ceilings equal the configuration values (100, 100, 200, 1 024, 64) and the binding pass ceiling C14c targets is the documented `_TRIP_BOUND` = 100, below the inert 200; the C14c/C14d configuration overrides are sound (`C14C_N_TRADES` above the default ring, `C14D_MAX_ICEBERGS` in $[2, 1024)$); the dataset precondition on every draw (section 4); every requested draw ran; and the eager comparisons — on the first `EAGER_PER_PART` **valid** draws of every part, a missing one recorded as a failed `eager_jit` assertion.

**Reported, not asserted:** C14c's cut point, trade count and full outcome per draw (whole), the live-iceberg count after C14d's sweep, the battery status counts per part, and the capacity peaks (which, here and only here, exceed the I8 envelope by design).

In [ ]:
import traceback
from collections import defaultdict

from omiecid_simulator.jaxob.jaxob_constants import TradesFeat
from tests.omiecid.reference import omie_invariants as inv

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Capacity edges, one part per ceiling",
    known_result="C14a: the 101st resting add raises book_full and wipes every worst-price order into the evicted "
                 "bucket, no trade; C14a′ (recorded, not certified; rebuilt r2, full only at regeneration): no flag "
                 "site meets a full side, the regenerations replace their own consumed rows, no flag and nothing "
                 "evicted or lost; C14b: after 100 trades the ring keeps only a burst's final trade, in its last row, "
                 "detectably (saturation flag), while the engine reports the true burst quantity (I6 and I2 violated "
                 "exactly on the burst); C14c: an aggressor needing > 100 passes (hop bound inside the 160 margin) "
                 "raises truncated, the loop terminates, I2 closes, the full outcome consistent with its fills; "
                 "C14c-below: hop bound < 100, no truncation, the exact walk; C14d: an iceberg beyond the slot cap "
                 "raises truncated at registration, leaves the side channel unchanged, its hidden remainder "
                 "untracked, the survivors intact; C14e: 65 slices leave truncated false, 66 set it at registration "
                 "with no trade; S2 controls: with the last rows occupied, a crossing NON on a full own side loses "
                 "only the documented worst-level wipe, and an iceberg regenerates and is reached; excluded "
                 "invariants violated exactly as stated",
    oracle_kind="exact-parametric on documented behaviour, plus the battery with per-part exclusions asserted "
                "expected-violated",
    quotes=QUOTES, citations=CITATIONS, constructions=(), requested_draws=REQUESTED, runtime_forecast_s=900.0,
    started_at=T0,
    downstream="I8 in every other scenario (C00-C13, C15-C17), which relies on these ceilings behaving as documented "
               "so that a breach is visible; the replay fidelity of busy real contracts (OQ-10); any consumer of the "
               "book or the trade ring at a ceiling",
)

RULE = {
    "a": "OQ-10 Decision impact (2026-06-11 accepted limitation); review §5 item 3; book-depth-cap-finding.md",
    "a_prime": "recorded, not certified: review V3-14; close-out record F1; book-depth-cap-finding.md correction 1a "
               "(regenerations are not flag sites); OQ-20 (finding F16)",
    "b": "OQ-15 (last-row writes; diff-based fill count collapses); review §5 item 2, §11.1; finding F4 (corrected)",
    "c": "§4.2.1 Rules 2 and 4; §4.2; §6.1; review §5 item 1, its correction note (100 passes) and V3-15 "
         "(findings F6, F7)",
    "c_below": "the capacity-validity precondition (sound hop bound < 100 passes: truncated == False); §4.2; §6.1 (F6)",
    "lr_non": "OMIE-JAXOB-LOOKUP-MISS-1 (S2, a miss is a miss); book-depth-cap-finding.md correction 1a (the wipe on "
              "any dispatch against a full own side); OQ-10 (finding F17)",
    "lr_ice": "OMIE-JAXOB-LOOKUP-MISS-1 (S2, no stranded iceberg); §4.2; §6.1 (finding F17)",
    "d": "jax_constants.py §B1 (slot overflow: meta unchanged, truncated); OQ-20; close-out audit (findings F8, F9)",
    "e65": "jax_constants.py §B2 (MAX_SLICES_PER_REGEN); the capacity-validity precondition, slice ceiling (F13)",
    "e66": "jax_constants.py §B2 (MAX_SLICES_PER_REGEN); the capacity-validity precondition, slice ceiling (F13)",
}
R_EXCL = "review V3-15: each excluded invariant is reported expected-violated and its violation asserted"
R_BAT = "the battery (RB, I1-I9): every non-excluded check at the status section 5 states"
R_I8 = "I8 and the capacity-validity precondition (five ceilings, slice margin re-checked from the tape; F12)"
R_DATA = "dataset precondition (section 4); §2.3"
R_CEIL = ("the configuration: 100 per side, 100 trades, max_match_iterations 200 (inert), 1024 icebergs, 64 slices; "
          "the capacity-validity precondition (v2.4): the binding pass ceiling is _TRIP_BOUND = 100")
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"

TR_P, TR_Q, TR_PASS, TR_AGR, TR_SEC = (TradesFeat.P.value, TradesFeat.Q.value, TradesFeat.PASS_OID.value,
                                       TradesFeat.AGRS_OID.value, TradesFeat.SEC.value)


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   unit="other", detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1],
                   rule=R_DATA)
        return None


def differing(exp, obs):
    if isinstance(exp, dict) and isinstance(obs, dict):
        return sorted(k for k in set(exp) | set(obs) if exp.get(k) != obs.get(k))
    return [] if exp == obs else ["value"]


def assert_json(name, seed, expected, observed, rule, unit="other"):
    ok = canon(expected) == canon(observed)
    return cert.check(name, ok, seed=seed, expected=canon(expected), observed=canon(observed), unit=unit,
                      detail="" if ok else "differs on: " + ", ".join(map(str, differing(expected, observed))),
                      rule=rule)


# ---- observed counterparts of section 5 ---------------------------------------------------------------------------
def observe_outcome(rec, keys):
    fin = rec.final_snapshot
    full = {
        "accepted": sorted(int(k) for k in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "fills": [[int(f.submission), int(f.row_index), int(f.price_int), int(f.qty_int), int(f.buyer_oid),
                   int(f.seller_oid)] for f in rec.fills],
        "filled": nz(rec.filled),
        "still_resting": nz(rec.still_resting),
        "hidden_remaining": nz(rec.hidden_remaining),
        "evicted": nz(rec.evicted),
        "cancelled": {str(int(k)): [[str(c.cause), int(c.qty_int)] for c in cs]
                      for k, cs in sorted(rec.cancelled_or_expired.items(), key=lambda kv: int(kv[0])) if cs},
        "final_book": {"BUY": sorted([int(r[0]), int(r[1]), int(r[2])] for r in fin.resting(BUY)),
                       "SELL": sorted([int(r[0]), int(r[1]), int(r[2])] for r in fin.resting(SELL))},
        "reported_qty": [int(round(float(er.trade_qty_mw) / float(CONFIG.qty_tick))) for er in rec.engine_reports],
    }
    return {k: full[k] for k in keys}


def observe_flags(rec):
    posts = rec.post_snapshots
    return {"book_full_posts": [s for s, sn in enumerate(posts) if bool(sn.book_full)],
            "truncated_posts": [s for s, sn in enumerate(posts) if bool(sn.truncated)],
            "trade_buffer_saturated": bool(rec.flags.get("trade_buffer_saturated", False))}


def ring_rows(snap):
    # The ring after the run, row by row: [price, qty, buyer oid, seller oid] (sign of Q: the aggressor bought if < 0).
    out = []
    for row in np.asarray(snap.trades):
        if int(row[TR_SEC]) == -1:
            continue
        q, pas, agr = int(row[TR_Q]), int(row[TR_PASS]), int(row[TR_AGR])
        buyer, seller = (agr, pas) if q < 0 else (pas, agr)
        out.append([int(row[TR_P]), abs(q), buyer, seller])
    return out


def observe_extra(part, rec, draw):
    prm, posts, pres = draw.params, rec.post_snapshots, rec.pre_snapshots
    if part in ("a", "a_prime"):
        return {"full_side_rows_posts": [len(sn.resting(prm["S"])) for sn in posts]}
    if part in ("lr_non", "lr_ice"):
        fin = rec.final_snapshot
        S = prm["S"]
        out = {"full_side_rows_posts": [len(sn.resting(S)) for sn in posts],
               "last_rows_resting": {"S": [prm["last_row_S"],
                                           any(int(r[2]) == prm["last_row_S"] for r in fin.resting(S))]}}
        if part == "lr_non":
            O = opp(S)
            out["other_side_rows_posts"] = [len(sn.resting(O)) for sn in posts]
            out["last_rows_resting"]["O"] = [prm["last_row_O"],
                                             any(int(r[2]) == prm["last_row_O"] for r in fin.resting(O))]
        return out
    if part == "b":
        return {"ring": ring_rows(rec.final_snapshot), "rows_used_posts": [int(sn.trade_rows_used()) for sn in posts]}
    if part == "d":
        M = prm["M"]
        pre, post = pres[M], posts[M]
        a, b = np.asarray(pre.iceberg_meta), np.asarray(post.iceberg_meta)
        same = int(pre.n_active) == int(post.n_active) and a.shape == b.shape and np.array_equal(a, b)
        return {"n_active_posts_through_overflow": [int(sn.n_active) for sn in posts[:M + 1]],
                "meta_unchanged_at_overflow": bool(same)}
    return {}


def run_checks(rec, e):
    return {
        "RB": inv.check_record(rec),
        "I1": inv.check_i1_volume_balance(rec),
        "I2": inv.check_i2_quantity_conservation(rec, allow_eviction=e["allow_eviction"]),
        "I3": inv.check_i3_uncrossed_on_return(rec),
        "I4": inv.check_i4_priority(rec),
        "I5": inv.check_i5_price_setter(rec),
        "I6": inv.check_i6_settlement(rec),
        "I7": inv.check_i7_no_manufactured_liquidity(rec),
        "I8": inv.check_i8_capacity_headroom(rec, envelope=ENVELOPE, allow_breach=True),
        "I9": inv.check_i9_non_vacuity(rec, expects_trade=e["expects_trade"]),
    }


def folded(res, e, label):
    # The report folded into the invariant table: each excluded check shown not_applicable (asserted separately).
    out = []
    for cid in BATTERY_IDS:
        r = res[cid]
        if cid in e["excluded"]:
            r = inv.CheckResult(check_id=cid, status="not_applicable", violations=(), unobserved=tuple(r.unobserved),
                                evidence={"reason": f"excluded on {label}: expected-violated, asserted separately"})
        out.append(r)
    return inv.BatteryReport(results=tuple(out))


def violation_sig(r):
    return sorted(([None if v.oid is None else int(v.oid), str(v.field), inv_json(v.expected), inv_json(v.observed)]
                   for v in r.violations), key=canon)


def inv_json(x):
    try:
        return json.loads(canon(x))
    except (TypeError, ValueError):
        return repr(x)


def messages(r):
    return [f"{v.field}: {v.message}" for v in r.violations]


# ---- the excluded checks -------------------------------------------------------------------------------------------
def check_i8(label, seed, rec, res, e):
    r = res["I8"]
    ev = dict(r.evidence)
    obs_ax, mar, fl = dict(ev.get("observed", {})), dict(ev.get("margins", {})), dict(ev.get("flags", {}))
    battery_breach = sorted([a for a in obs_ax if a in mar and int(obs_ax[a]) > int(mar[a])] +
                            [k for k, v in fl.items() if v])
    ms = max_slices(rec.tape)
    breach = sorted(battery_breach + (["slices"] if ms > SLICE_MARGIN else []))
    values = {**{a: int(v) for a, v in obs_ax.items()}, "slices": int(ms)}
    want_status = "not_applicable" if battery_breach else "pass"
    req, allowed, pinned = set(e["i8"]["required"]), set(e["i8"]["allowed"]), e["i8"]["pinned"]
    ok = (req <= set(breach) <= allowed and all(values.get(k) == v for k, v in pinned.items())
          and r.status == want_status)
    return cert.check(
        f"{label}: I8 excluded — complete breach set as stated (asserted)", ok, seed=seed,
        expected=canon({"breach_superset_of": sorted(req), "breach_subset_of": sorted(allowed), "pinned": pinned,
                        "battery_status": want_status}),
        observed=canon({"breach": breach, "values": values, "flags": fl, "margins": mar, "slice_margin": SLICE_MARGIN,
                        "battery_status": r.status, "battery_breaches": list(ev.get("breaches", []))}),
        unit="other", detail="" if ok else "breach set, pinned value or I8 status departs", rule=R_I8)


def status_ok(observed, expected):
    # Judging rule (OMIE-CERT-RERUN-1): not_applicable is vacuously satisfied wherever pass is expected.
    return observed == expected or (expected in ("pass", NA) and observed in ("pass", NA))


def check_status_spec(label, seed, cid, r, spec):
    obs = {"status": r.status}
    exp = {"status": spec["status"]}
    if "violations" in spec:
        obs["violations"] = violation_sig(r)
        exp["violations"] = sorted(spec["violations"], key=canon)
    ok = status_ok(obs["status"], exp["status"]) and obs.get("violations") == exp.get("violations")
    what = "expected-violated as stated" if spec["status"] == "fail" else "derived status (finding F4)"
    return cert.check(f"{label}: {cid} excluded — {what} (asserted)", ok, seed=seed, expected=canon(exp),
                      observed=canon({**obs, "messages": messages(r)}), unit="other",
                      detail="" if ok else "status or violation set departs", rule=R_EXCL)


def check_i6_burst(label, seed, r6, spec):
    # C14b I6 (finding F4, corrected): fail; the burst submission's quantity violation required; besides it, only a
    # settlement-value violation (field engine_reports) on that same submission is admitted.
    s_b = spec["submission"]
    sigs = sorted(([None if v.submission is None else int(v.submission), str(v.field), inv_json(v.expected),
                    inv_json(v.observed)] for v in r6.violations), key=canon)
    qty = [s_b, "engine_reports", spec["qty_violation"][2], spec["qty_violation"][3]]
    extra = [g for g in sigs if g != qty]
    settlement_like = all(g[0] == s_b and g[1] == "engine_reports" and isinstance(g[2], float) for g in extra)
    ok = r6.status == "fail" and qty in sigs and sigs.count(qty) == 1 and settlement_like and len(extra) <= 1
    return cert.check(
        f"{label}: I6 excluded — fails on exactly the burst submission, report T_b against rows r_b (asserted)", ok,
        seed=seed,
        expected=canon({"status": "fail", "required": qty,
                        "admitted": f"at most one settlement-value violation on submission {s_b}"}),
        observed=canon({"status": r6.status, "violations": sigs, "messages": messages(r6)}),
        unit="other", detail="" if ok else "status or violation set departs", rule=R_EXCL + "; finding F4")


def check_c_outcome(label, seed, rec, e):
    # C14c (r2): the full outcome, recorded whole, checked for consistency with the observed fills.
    w = e["walk"]
    agg, S, Q = w["agg"], w["S"], w["Q"]
    O = opp(S)
    obs = observe_outcome(rec, ("filled", "still_resting", "hidden_remaining", "final_book", "reported_qty", "fills"))
    by_oid = Counter()
    for f in rec.fills:
        by_oid[int(f.buyer_oid)] += int(f.qty_int)
        by_oid[int(f.seller_oid)] += int(f.qty_int)
    f_agg = int(by_oid.get(agg, 0))
    ice = {int(k): v for k, v in w["ices"].items()}
    price_of = {**{j: (S, v["price"]) for j, v in ice.items()}, agg: (O, w["agg_price"])}
    rest = {int(k): int(v) for k, v in obs["still_resting"].items()}
    hid = {int(k): int(v) for k, v in obs["hidden_remaining"].items()}
    problems = []
    if obs["filled"] != nz(by_oid):
        problems.append("filled is not the per-order sum of the fills")
    rep = [0] * w["n_orders"]
    rep[w["agg_submission"]] = f_agg
    if obs["reported_qty"] != rep:
        problems.append(f"reported_qty is not {rep}")
    if rest.get(agg, 0) != Q - f_agg:
        problems.append(f"the aggressor rests {rest.get(agg, 0)}, not Q - filled = {Q - f_agg}")
    if hid.get(agg, 0):
        problems.append("the aggressor has hidden quantity")
    if not set(rest) <= set(price_of):
        problems.append(f"orders resting outside the construction: {sorted(set(rest) - set(price_of))}")
    if not set(hid) <= set(ice):
        problems.append(f"hidden quantity outside the icebergs: {sorted(set(hid) - set(ice))}")
    for j, v in ice.items():
        if rest.get(j, 0) > v["p"]:
            problems.append(f"iceberg {j} rests {rest[j]} > its peak {v['p']}")
    want_book = book_json([(price_of[o][0], price_of[o][1], q, o) for o, q in rest.items() if o in price_of])
    if obs["final_book"] != want_book:
        problems.append("final_book is not still_resting at each order's declared side and price")
    ok = not problems
    return cert.check(
        f"{label}: full outcome recorded and consistent with the observed fills", ok, seed=seed,
        expected=canon({"rule": "filled = per-order sum of fills; reported_qty = [0,0,0,0,filled(agg)]; the aggressor "
                                "rests Q - filled(agg), no hidden; each iceberg rests <= its peak; hidden on icebergs "
                                "only; final_book = still_resting at declared side and price", "Q": Q}),
        observed=canon({**obs, "problems": problems}), unit="other", detail="; ".join(problems), rule=RULE["c"])


def check_c_i3(label, seed, rec, r3, e):
    w = e["walk"]
    fin = rec.final_snapshot
    bb, ba = fin.best_bid(), fin.best_ask()
    visible_cross = bb is not None and ba is not None and bb >= ba
    agg_rest = int(fin.visible_qty(w["agg"], w["agg_side"]))
    consumed = sorted(int(j) for j in w["ices"]
                      if int(rec.hidden_remaining.get(int(j), 0)) > 0 and int(fin.visible_qty(int(j), w["S"])) == 0
                      and agg_rest > 0)
    points = sorted({canon([str(v.field), v.submission]) for v in r3.violations})
    allowed_pts = {canon(["post_snapshots", w["agg_submission"]]), canon(["final_snapshot", None])}
    confined = set(points) <= allowed_pts
    departure = visible_cross or bool(consumed)
    consistent = (r3.status == "fail") == visible_cross
    ok = departure and confined and consistent
    return cert.check(
        f"{label}: I3 excluded — Rule-4 departure present, I3 fails iff visibly crossed, only on the truncating "
        f"submission (asserted)", ok, seed=seed,
        expected=canon({"rule4_departure": True, "i3_fail_iff_visible_cross": True,
                        "i3_violation_points_within": sorted(allowed_pts)}),
        observed=canon({"rule4_departure": departure, "visible_cross": visible_cross, "best_bid": bb, "best_ask": ba,
                        "aggressor_resting": agg_rest, "consumed_slots": consumed, "i3_status": r3.status,
                        "i3_violation_points": points, "i3_messages": messages(r3)}),
        unit="other", detail="" if ok else "departure absent, I3 inconsistent, or an I3 violation before the truncating "
                                           "submission", rule=RULE["c"] + "; " + R_EXCL)


def check_walk(label, seed, rec, e):
    w = e["walk"]
    seq, ices, agg, S = w["sequence"], w["ices"], w["agg"], w["S"]
    F = [[int(f.price_int), int(f.qty_int), int(f.buyer_oid), int(f.seller_oid)] for f in rec.fills]
    subs = sorted({int(f.submission) for f in rec.fills})
    ice_leg = (lambda row: row[3]) if S == SELL else (lambda row: row[2])
    agg_leg = (lambda row: row[2]) if S == SELL else (lambda row: row[3])
    idx = 0
    while idx < len(F) and idx < len(seq) and F[idx] == seq[idx]:
        idx += 1
    tail = F[idx:]
    problems = []
    if not F:
        problems.append("no fill")
    elif idx == 0:
        problems.append("the first fill is not the walk's first slice")
    if subs and subs != [w["agg_submission"]]:
        problems.append(f"fills on submissions {subs}, not only the aggressor's {w['agg_submission']}")
    min_rank = ices[str(ice_leg(seq[idx]))]["rank"] if idx < len(seq) else len(ices)
    last_rank = None
    for row in tail:
        info = ices.get(str(ice_leg(row)))
        if info is None or agg_leg(row) != agg:
            problems.append(f"tail fill {row}: not between the aggressor and one of the icebergs")
            continue
        if row[0] != info["price"] or not 1 <= row[1] <= info["p"]:
            problems.append(f"tail fill {row}: price or quantity departs (iceberg price {info['price']}, peak {info['p']})")
        if info["rank"] < min_rank or (last_rank is not None and info["rank"] <= last_rank):
            problems.append(f"tail fill {row}: out of price order (rank {info['rank']})")
        last_rank = info["rank"]
    filled_agg = int(rec.filled.get(agg, 0))
    if not 0 < filled_agg < w["Q"]:
        problems.append(f"aggressor filled {filled_agg}, not in (0, {w['Q']})")
    ok = not problems
    return cert.check(
        f"{label}: walk — the documented walk up to the cut point, then at most one visible slice per later iceberg in "
        f"price order; 0 < filled < Q", ok, seed=seed,
        expected=canon({"walk": seq, "Q": w["Q"], "submission": w["agg_submission"],
                        "rule": "fills == walk[:cut] + tail, tail strictly increasing in price rank from the next "
                                "slice's iceberg, each fill at that iceberg's price for <= its peak; 0 < filled < Q"}),
        observed=canon({"fills": F, "cut": idx, "tail": tail, "filled_aggressor": filled_agg, "submissions": subs,
                        "problems": problems}),
        unit="other", detail="; ".join(problems), rule=RULE["c"])


# ---- one draw --------------------------------------------------------------------------------------------------
STATUS = defaultdict(Counter)        # "C14a:I2" -> status counts (actual results, excluded ones included)
C_REPORT = []                        # C14c: cut point and trade count per draw (reported)
D_REPORT = []                        # C14d: live icebergs after the sweep (reported)


def check_draw(part, seed, draw, e, rec, res):
    label = f"{LABEL[part]} seed {seed}" + (" (recorded, not certified)" if part == "a_prime" else "")
    rule = RULE[part]
    for cid, r in res.items():
        STATUS[f"{LABEL[part]}:{cid}"][r.status] += 1
    ok = [assert_json(f"{label}: engine outcome", seed, e["outcome"], observe_outcome(rec, e["outcome"].keys()), rule),
          assert_json(f"{label}: flags (book_full / truncated posts, ring saturated)", seed, e["flags"],
                      observe_flags(rec), rule)]
    if e["extra"]:
        ok.append(assert_json(f"{label}: part detail", seed, e["extra"], observe_extra(part, rec, draw), rule))
    if part == "c":
        ok.append(check_walk(label, seed, rec, e))
        ok.append(check_c_outcome(label, seed, rec, e))
        C_REPORT.append({"seed": seed, "trades": len(rec.fills), "walk_length": len(e["walk"]["sequence"]),
                         "filled_aggressor": int(rec.filled.get(e["walk"]["agg"], 0)), "Q": e["walk"]["Q"],
                         "outcome": observe_outcome(rec, ("filled", "still_resting", "hidden_remaining",
                                                          "final_book", "reported_qty"))})
    if part == "d":
        D_REPORT.append({"seed": seed, "n_active_final": int(rec.final_snapshot.n_active)})
    bat_obs = {cid: res[cid].status for cid in e["battery"]}
    bad = {cid: messages(res[cid]) for cid in e["battery"] if not status_ok(res[cid].status, e["battery"][cid])}
    ok.append(cert.check(f"{label}: battery (non-excluded) at the stated statuses", not bad, seed=seed,
                         expected=canon(e["battery"]), observed=canon(bat_obs), unit="other",
                         detail="" if not bad else "departing checks and their violations: " + canon(bad), rule=R_BAT))
    for cid, spec in e["excluded"].items():
        if spec["kind"] == "i8":
            ok.append(check_i8(label, seed, rec, res, e))
        elif spec["kind"] == "status":
            ok.append(check_status_spec(label, seed, cid, res[cid], spec))
        elif spec["kind"] == "i6_burst":
            ok.append(check_i6_burst(label, seed, res[cid], spec))
        elif spec["kind"] == "c_i3":
            ok.append(check_c_i3(label, seed, rec, res["I3"], e))
        else:
            ok.append(cert.check(f"{label}: {cid} excluded — unknown spec kind", False, seed=seed,
                                 expected="a known kind", observed=str(spec.get("kind")), unit="other", rule=R_EXCL))
    return all(ok)


# ---- the run ----------------------------------------------------------------------------------------------------
wired = callable(globals().get("run_part"))
cert.check("section 6 wired (run_part defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_DATA)
cert.check("the five configured ceilings are the documented configuration values (max_match_iterations 200 inert)",
           list(CEILINGS.values()) == CONFIGURED_CEILINGS,
           expected=canon(CONFIGURED_CEILINGS), observed=canon(list(CEILINGS.values())), unit="other", rule=R_CEIL)
cert.check("the binding pass ceiling C14c targets is the documented _TRIP_BOUND (100), below the configured 200",
           PASS_BOUND == 100 and PASS_BOUND < ITER and DOCUMENTED_CEILINGS[2] == PASS_BOUND,
           expected=canon({"PASS_BOUND": 100, "max_match_iterations": f"> {100}"}),
           observed=canon({"PASS_BOUND": PASS_BOUND, "max_match_iterations": ITER}), unit="other", rule=R_CEIL)
_ovr = {"C14C_N_TRADES": int(C14C_N_TRADES), "C14D_MAX_ICEBERGS": int(C14D_MAX_ICEBERGS)}
cert.check("C14c/C14d configuration overrides are sound (ring above the default; slot cap in [2, 1024))",
           int(C14C_N_TRADES) > RING and 2 <= int(C14D_MAX_ICEBERGS) < int(MARKET.max_icebergs),
           expected=canon({"C14C_N_TRADES": f"> {RING}", "C14D_MAX_ICEBERGS": f"in [2, {int(MARKET.max_icebergs)})"}),
           observed=canon(_ovr), unit="other", rule="findings F6, F8")
cert.check("dataset: every draw is as section 4 describes (precondition)", not PRE_BAD, expected=canon({}),
           observed=canon(PRE_BAD), unit="other", detail=f"{len(PRE_BAD)} draws depart" if PRE_BAD else
           f"{len(DRAWN)} draws", rule=R_DATA)

EAGER_DONE = defaultdict(list)
PER_DRAW = []
RAN = Counter()
if wired:
    for part in PARTS:
        for k, seed in enumerate(SEEDS[part]):
            draw, e = DRAWN[(part, seed)], EXPECTED[(part, seed)]
            label = f"{LABEL[part]} seed {seed}"
            rec = guarded(f"{label}: run_part (jit)", seed, run_part, draw, jit=True)
            if rec is None:
                cert.discard(seed, "run_part raised (recorded as a failed check)")
                continue
            RAN[part] += 1
            cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION})
            try:
                res = run_checks(rec, e)
                cert.add_battery(folded(res, e, LABEL[part]), seed=seed)
                PER_DRAW.append({"part": LABEL[part], "seed": seed, "ok": check_draw(part, seed, draw, e, rec, res)})
            except Exception as exc:  # noqa: BLE001
                cert.check(f"{label}: section 7 could not read the record", False, seed=seed, expected="readable",
                           observed=type(exc).__name__, unit="other",
                           detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1],
                           rule=R_DATA)
                PER_DRAW.append({"part": LABEL[part], "seed": seed, "ok": False})
            cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            # eager on the first EAGER_PER_PART VALID draws (precondition holds, jitted record returned)
            valid = f"{LABEL[part]} seed {seed}" not in PRE_BAD
            if valid and len(EAGER_DONE[part]) < EAGER_PER_PART:
                eg = guarded(f"{label}: run_part (eager)", seed, run_part, draw, jit=False)
                if eg is not None:
                    diff = list(inv.compare_records(eg, rec))
                    cert.set_eager_jit(seed, not diff, detail=", ".join(diff))
                    EAGER_DONE[part].append(seed)

# ---- aggregate assertions ---------------------------------------------------------------------------------------
cert.check("every requested draw ran", sum(RAN.values()) == REQUESTED, expected=canon({LABEL[p]: DRAWS for p in PARTS}),
           observed=canon({LABEL[p]: RAN[p] for p in PARTS}), unit="count", rule=R_DATA)
for _p in PARTS:
    if len(EAGER_DONE.get(_p, [])) < min(EAGER_PER_PART, DRAWS):
        cert.check(f"eager == jitted for {LABEL[_p]}", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"{len(EAGER_DONE.get(_p, []))} eager comparison(s) run for {LABEL[_p]}, "
                          f"{min(EAGER_PER_PART, DRAWS)} required", rule=R_EAGER)

# ---- reports (not asserted) -------------------------------------------------------------------------------------
_fail = [f"{r['part']} seed {r['seed']}" for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_DRAW)} draws checked, {len(_fail)} with a failed assertion"
                 + (": " + ", ".join(_fail) if _fail else ".")))
print(f"draws run per part: { {LABEL[p]: RAN[p] for p in PARTS} }")
print(f"eager comparisons run: { {LABEL[p]: v for p, v in EAGER_DONE.items()} }")
print(f"C14c cut points and full outcomes (reported, in full): {canon(C_REPORT)}")
print(f"C14d live icebergs after the sweep (reported): {D_REPORT}")
print(f"battery status counts, actual results (excluded checks included): { {k: dict(v) for k, v in STATUS.items()} }")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))

## 9. Why this certification failed

### 9.1 What failed

The notebook passed; its audit twin failed: test_C14_audit.py::TestBrokenCopies::test_citation_tampering failed (48 of 49 checks passed): the check tampers the results' second citation by replacing 'worst level' with 'last row' and expects the citation judge to reject the copy, but after the K2 re-pointing the second citation (scenario-ladder.md v2.5, the capacity-validity precondition) no longer contains 'worst level', so the tamper changes nothing and the copy is still accepted. Every engine check, the quote check, the invariant battery and the citation check itself (all 12 citations resolve in the released repository) passed.

### 9.2 The smallest failing case

No counterexample fixture was recorded (the failure is not seed-specific).

### 9.3 Which rule it concerns

—

### 9.4 Where the fault most plausibly lies

Hypothesis: The audit test's broken-copy check, not the engine and not the notebook's own checks: the tamper targets the citation at position 1 and a fixed phrase, and the K2 change (citations re-pointed to shipped files, one dropped) moved the 'worst level' wording out of that position, so the mutation is a no-op. Resolving it needs a maintainer decision (re-point that citation to a fragment that contains the phrase, or have the check tamper a citation that does); nothing was re-authored here.

### 9.5 What it affects downstream

none: no other scenario reads C14's citations
